# Project 5 — From Static to Contextual (UD English EWT)
**Yellapu Dheeraj Babu (23CS10081)** · NLP CS60075 Assignment II

**This notebook is self-contained.** It holds the complete source code: every file of the project is one cell below,
grouped as *shared modules*, *Task 1*, *Task 2* and *Task 3*. Running it writes the code to a fresh folder, installs
the pinned packages, downloads the public data and runs every experiment of the report. **Nothing has to be uploaded.**

**How to run:** *Runtime → Change runtime type → GPU* (a T4 is enough), then *Runtime → Run all*.

| What is downloaded | Source (pinned) |
|---|---|
| UD English EWT | Hugging Face `universal-dependencies/universal_dependencies`, tag `ud2.18-loader-v2.3.0` |
| WordSim-353, Google analogies | gensim test data, commit `88d032b` |
| `bert-base-uncased` | Hugging Face `google-bert/bert-base-uncased`, revision `86b5e09` |

**Choose a mode in the next cell** (Task 2's 78 probes are the only long step):

| `MODE` | Task 2 probes | Approx. time on a T4 |
|---|---|---|
| `"full"` (default) | train all 78 probes, exactly as in the report | several hours |
| `"fast"` | train only rank 64 (26 probes) | ~1–2 h |
| `"pretrained"` | download our 78 trained probes from the project's GitHub repo, then evaluate | ~30 min |

Everything else (data, Task 1, BERT features, Task 3) runs in full in every mode. All seeds are fixed (`SEED = 42`).
Probe training is **resumable**: after a disconnect, *Run all* again and finished probes are skipped.

In [ ]:
MODE = "full"          # "full" | "fast" | "pretrained"  (see the table above)
PIN_TORCH = True       # True: install the exact pinned torch (reproducible); False: keep Colab's preinstalled torch (faster)
WITH_CONTROL = False   # True: also train the spec-literal Task 2 control (constant lr, last epoch; report Sec. 4.2)

In [ ]:
import os, sys, subprocess
PROJECT_DIR = "/content/project5" if os.path.isdir("/content") else os.path.abspath("project5")
for d in ["code/common", "code/task1", "code/task2", "code/task3"]:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)
os.chdir(PROJECT_DIR)

def run(*args):
    """Run a project script with live output; stop the notebook if it fails."""
    cmd = [sys.executable, *args]
    print("$ python", " ".join(args), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed: {' '.join(args)}")

print("project folder:", PROJECT_DIR)

## 1. Environment: pinned packages

In [ ]:
%%writefile requirements.txt
# Project 5 (UD English EWT) - Python 3.12.16
# Exact versions from the development env `nlp5` (frozen 2026-11-06 after Phase 0).
# torch was installed from PyPI as 2.15.0+cu132 (CUDA 13.2 build).
numpy==2.5.4
scipy==1.18.1
scikit-learn==1.9.1
pandas==3.1.1
matplotlib==3.11.2
tqdm==4.70.1
torch==2.15.0
transformers==5.22.1
datasets==5.1.0          # the UD repo is Parquet-only; script loaders were removed in datasets 4.0
huggingface_hub==2.4.0


In [ ]:
req = [l for l in open("requirements.txt").read().splitlines() if l.strip() and not l.startswith("#")]
if not PIN_TORCH:
    req = [r for r in req if not r.startswith("torch")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r.split("#")[0].strip() for r in req]], check=True)
if PIN_TORCH:
    # Colab preinstalls torchaudio/torchvision built for a different CUDA than the pinned torch; transformers imports
    # torchaudio whenever it is present and then fails ("PyTorch and TorchAudio were compiled with different CUDA
    # versions"). This project uses neither package, so remove them.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio", "torchvision"], check=False)
# end-to-end import check, including loading the BERT class that failed before
subprocess.run([sys.executable, "-c", "import torch, transformers, datasets, numpy; from transformers import BertModel; "
                "print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| transformers', "
                "transformers.__version__, '| datasets', datasets.__version__, '| numpy', numpy.__version__, '| BertModel OK')"],
               check=True)

## 2. Shared modules (`code/common/`)
Configuration and seeds, EWT loading, benchmarks, MST and tree utilities, bootstrap statistics, and the evaluation
suite used by **both** Task 1 and Task 3, so the two use an identical protocol. Each cell writes one file.

In [ ]:
%%writefile code/common/__init__.py
"""Shared utilities for Project 5 (data, benchmarks, trees, statistics, config)."""


In [ ]:
%%writefile code/common/benchmarks.py
"""WordSim-353 and Google analogy benchmarks: download, load, coverage, and evaluation.

The same functions are used for Task 1 (PPMI+SVD) and Task 3 (BERT-pooled) vectors, so the
protocol is identical by construction.

Vector interface: `vectors` is a float array [V, d], `w2i` maps lowercased word -> row.
"""
from __future__ import annotations

import hashlib
import urllib.request
from collections import OrderedDict
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

from . import config as C


# --------------------------------------------------------------------------- download / load
def _download(url: str, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"[benchmarks] downloading {url}")
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return hashlib.sha256(path.read_bytes()).hexdigest()


def download_benchmarks() -> dict[str, str]:
    """Download both benchmark files (if missing). Returns sha256 checksums."""
    return {
        "wordsim353": _download(C.WORDSIM_URL, C.WORDSIM_PATH),
        "google_analogy": _download(C.ANALOGY_URL, C.ANALOGY_PATH),
    }


def load_wordsim(path: Path = C.WORDSIM_PATH) -> list[tuple[str, str, float]]:
    """353 (word1, word2, human_score) triples, lowercased, file order preserved."""
    pairs = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        w1, w2, score = line.split("\t")[:3]
        pairs.append((w1.lower(), w2.lower(), float(score)))
    return pairs


def load_analogies(path: Path = C.ANALOGY_PATH) -> "OrderedDict[str, list[tuple[str, str, str, str]]]":
    """Section name -> list of (a, b, c, d) meaning a:b :: c:d. Lowercased (gensim default)."""
    sections: OrderedDict[str, list] = OrderedDict()
    current = None
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        if line.startswith(":"):
            current = line[1:].strip()
            sections[current] = []
        else:
            a, b, c, d = line.lower().split()
            sections[current].append((a, b, c, d))
    return sections


def is_syntactic(section: str) -> bool:
    return section.startswith("gram")


# --------------------------------------------------------------------------- coverage
def wordsim_coverage(pairs, vocab) -> dict:
    covered = sum(1 for a, b, _ in pairs if a in vocab and b in vocab)
    words = {w for a, b, _ in pairs for w in (a, b)}
    return dict(n_pairs=len(pairs), n_covered=covered, pair_coverage=covered / len(pairs),
                n_words=len(words), word_coverage=sum(w in vocab for w in words) / len(words))


def analogy_coverage(sections, vocab) -> dict:
    out, tot, cov = {}, 0, 0
    for name, qs in sections.items():
        c = sum(1 for q in qs if all(w in vocab for w in q))
        out[name] = dict(n=len(qs), covered=c, coverage=c / len(qs))
        tot += len(qs)
        cov += c
    return dict(n_questions=tot, n_covered=cov, coverage=cov / tot, per_section=out)


# --------------------------------------------------------------------------- evaluation
def _unit(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, 1e-12)


def evaluate_wordsim(vectors: np.ndarray, w2i: dict[str, int], pairs=None,
                     oov_policy: str = "median") -> dict:
    """Spearman between model cosine similarities and human scores.

    * covered: only pairs whose two words are in the vocabulary (main number).
    * full: all 353 pairs; OOV pairs get an imputed similarity. Default "median" (the median
      of the model's covered similarities, i.e. a neutral rank). "zero" uses 0. Zero is NOT
      neutral for anisotropic vectors like BERT, where nearly all cosines are > 0, so the
      default is "median".
    """
    pairs = pairs if pairs is not None else load_wordsim()
    U = _unit(vectors)
    sims, human, covered_mask, records = [], [], [], []
    for a, b, h in pairs:
        ok = a in w2i and b in w2i
        s = float(U[w2i[a]] @ U[w2i[b]]) if ok else np.nan
        sims.append(s)
        human.append(h)
        covered_mask.append(ok)
        records.append(dict(word1=a, word2=b, human=h, model=s, covered=ok))
    sims, human, covered_mask = np.array(sims), np.array(human), np.array(covered_mask)

    rho_cov = spearmanr(sims[covered_mask], human[covered_mask]).statistic
    fill = 0.0 if oov_policy == "zero" else float(np.median(sims[covered_mask]))
    full = np.where(covered_mask, sims, fill)
    rho_full = spearmanr(full, human).statistic
    return dict(spearman_covered=float(rho_cov), spearman_full=float(rho_full),
                n_covered=int(covered_mask.sum()), n_total=len(pairs), oov_policy=oov_policy,
                records=records)


def evaluate_analogy(vectors: np.ndarray, w2i: dict[str, int], sections=None,
                     batch_size: int = 512) -> dict:
    """3CosAdd / vector offset: argmax_x cos(x, b - a + c) over the vocabulary, excluding a, b, c.

    Vectors are unit-normalised first (same as gensim's most_similar). Questions with any OOV
    word count as wrong in the "full" accuracy and are skipped in the "covered" accuracy.
    """
    sections = sections if sections is not None else load_analogies()
    U = _unit(vectors)
    i2w = {i: w for w, i in w2i.items()}
    per_section, examples = {}, []
    tot_n = tot_cov = tot_correct = 0
    agg = {"semantic": [0, 0, 0], "syntactic": [0, 0, 0]}  # n, covered, correct

    for name, qs in sections.items():
        cov = [q for q in qs if all(w in w2i for w in q)]
        correct = 0
        for start in range(0, len(cov), batch_size):
            batch = cov[start:start + batch_size]
            ia = np.array([w2i[q[0]] for q in batch])
            ib = np.array([w2i[q[1]] for q in batch])
            ic = np.array([w2i[q[2]] for q in batch])
            target = U[ib] - U[ia] + U[ic]  # [B, d]
            scores = target @ U.T  # [B, V]; argmax unaffected by |target|
            rows = np.arange(len(batch))
            for idx in (ia, ib, ic):
                scores[rows, idx] = -np.inf
            pred = scores.argmax(1)
            for q, p in zip(batch, pred):
                hit = i2w[int(p)] == q[3]
                correct += hit
                if len(examples) < 20000:  # all covered questions (5,727) are kept
                    examples.append(dict(section=name, a=q[0], b=q[1], c=q[2], gold=q[3],
                                         pred=i2w[int(p)], correct=bool(hit)))
        per_section[name] = dict(n=len(qs), covered=len(cov), correct=correct,
                                 acc_covered=correct / len(cov) if cov else float("nan"),
                                 acc_full=correct / len(qs))
        kind = "syntactic" if is_syntactic(name) else "semantic"
        agg[kind][0] += len(qs)
        agg[kind][1] += len(cov)
        agg[kind][2] += correct
        tot_n += len(qs)
        tot_cov += len(cov)
        tot_correct += correct

    def _acc(c, d):
        return c / d if d else float("nan")

    return dict(
        acc_covered=_acc(tot_correct, tot_cov), acc_full=_acc(tot_correct, tot_n),
        n_total=tot_n, n_covered=tot_cov, n_correct=tot_correct,
        by_kind={k: dict(n=v[0], covered=v[1], correct=v[2], acc_covered=_acc(v[2], v[1]),
                         acc_full=_acc(v[2], v[0])) for k, v in agg.items()},
        per_section=per_section, examples=examples,
    )


In [ ]:
%%writefile code/common/config.py
"""Central configuration: paths, pinned data/model sources, seeds and constants.

Every task imports from here, so all pinned versions and hyper-parameters live in one place.
"""
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np

# --------------------------------------------------------------------------- paths
# code/common/config.py -> project root is two levels above code/
PROJECT_ROOT = Path(os.environ.get("P5_ROOT", Path(__file__).resolve().parents[2]))
# Treebank: the assigned dataset is UD English EWT (default). The two other datasets listed for
# Project 5 can be selected with P5_TREEBANK=en_gum or P5_TREEBANK=en_lines (same HF loader and
# schema); they get their own data/<name>/ and outputs_<name>/ folders so nothing is overwritten.
TREEBANK = os.environ.get("P5_TREEBANK", "en_ewt")
assert TREEBANK in ("en_ewt", "en_gum", "en_lines"), f"unknown treebank {TREEBANK}"
_TB_SHORT = TREEBANK.removeprefix("en_")
DATA_DIR = PROJECT_ROOT / "data"
EWT_DIR = DATA_DIR / _TB_SHORT  # (name kept for backward compatibility: the folder of the selected treebank)
BENCH_DIR = DATA_DIR / "benchmarks"
OUTPUT_DIR = PROJECT_ROOT / ("outputs" if TREEBANK == "en_ewt" else f"outputs_{_TB_SHORT}")
SHARED_OUT = OUTPUT_DIR / "shared"
TASK1_OUT = OUTPUT_DIR / "task1"
TASK2_OUT = OUTPUT_DIR / "task2"
TASK3_OUT = OUTPUT_DIR / "task3"
# cached BERT word features (~5 GB); override with P5_FEATURE_DIR, e.g. a local scratch disk
FEATURE_DIR = Path(os.environ.get("P5_FEATURE_DIR", TASK2_OUT / "features"))

# --------------------------------------------------------------------------- seeds
SEED = 42


def set_seed(seed: int = SEED) -> None:
    """Fix python / numpy / torch RNGs (torch only if it is installed)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# --------------------------------------------------------------------------- dataset (UD English EWT)
# HF repo is Parquet-only (loader v2.x, needs datasets>=4.0). We pin an immutable *tag*
# (UD release 2.18, loader schema v2.3.0: upos = strings, head = ints, MWT surface forms
# excluded from `tokens`). Fallbacks are tried in order if the tag cannot be resolved.
UD_REPO = "universal-dependencies/universal_dependencies"
UD_CONFIG = TREEBANK
UD_REVISIONS = ["ud2.18-loader-v2.3.0", "2.18", "main"]
SPLITS = ["train", "dev", "test"]

# --------------------------------------------------------------------------- benchmarks
# gensim's test-data copies, pinned to a fixed commit for reproducibility.
_GENSIM_COMMIT = "88d032bce7772939a6ca051d8e30701b4eb59c1e"
_GENSIM_RAW = f"https://raw.githubusercontent.com/piskvorky/gensim/{_GENSIM_COMMIT}/gensim/test/test_data"
WORDSIM_URL = f"{_GENSIM_RAW}/wordsim353.tsv"
ANALOGY_URL = f"{_GENSIM_RAW}/questions-words.txt"
WORDSIM_PATH = BENCH_DIR / "wordsim353.tsv"
ANALOGY_PATH = BENCH_DIR / "questions-words.txt"

# --------------------------------------------------------------------------- model
BERT_MODEL = "google-bert/bert-base-uncased"  # == "bert-base-uncased" on the HF Hub
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
BERT_MAX_WORDPIECES = 510  # 512 minus [CLS] and [SEP]
BERT_NUM_LAYERS = 12  # hidden_states[0] = embeddings, [1..12] = transformer layers
BERT_DIM = 768

# --------------------------------------------------------------------------- Task 1
WINDOW = 5  # symmetric co-occurrence window (+-5)
SVD_DIM = 100
MIN_COUNT = 1  # keep every train word type (maximises benchmark coverage)

# --------------------------------------------------------------------------- 20-sentence MST check (T1 + T3)
MST_SAMPLE_SIZE = 20
MST_MIN_WORDS = 5  # non-punctuation words
MST_MAX_WORDS = 40
MST_SAMPLE_PATH = SHARED_OUT / "mst_sample_ids.json"

# --------------------------------------------------------------------------- Task 2 (Hewitt & Manning 2019)
PROBE_LAYERS = list(range(0, BERT_NUM_LAYERS + 1))  # 0 = embedding baseline, 1..12 required
PROBE_RANKS = [32, 64, 128]
PROBE_LR = 1e-3
PROBE_BATCH_SIZE = 20
PROBE_EPOCHS = 30
SPEARMAN_MIN_LEN = 5  # macro-average Spearman over sentence lengths 5..50 (as in the paper)
SPEARMAN_MAX_LEN = 50

# --------------------------------------------------------------------------- statistics
BOOTSTRAP_RESAMPLES = 1000


def ensure_dirs() -> None:
    for d in (DATA_DIR, EWT_DIR, BENCH_DIR, SHARED_OUT, TASK1_OUT, TASK2_OUT, TASK3_OUT):
        d.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile code/common/data.py
"""UD English EWT loading + gold tree utilities shared by all three tasks.

Data flow:
    HF Hub (Parquet, pinned tag) --download_ewt()--> data/ewt/{split}.jsonl --load_ewt()--> list[Sentence]

Conventions
-----------
* Word indices are 0-based inside python; `heads` keeps the CoNLL-U convention
  (1-based index of the head word, 0 = ROOT).
* `tokens` are the *syntactic words* (multi-word-token surface forms such as "don't"
  are NOT included; only "do" + "n't"), so `heads` index directly into `tokens`.
* Punctuation = UPOS "PUNCT" (decision D4 in plan.md).
"""
from __future__ import annotations

import json
from collections import Counter, deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Iterable

import numpy as np

from . import config as C


@dataclass
class Sentence:
    sent_id: str
    text: str
    tokens: list[str]
    upos: list[str]
    xpos: list[str]
    heads: list[int]  # 1-based head index, 0 = root
    deprels: list[str]
    n_mwt: int = 0
    n_empty: int = 0
    lower: list[str] = field(default_factory=list)  # lowercased tokens (decision D3)

    def __post_init__(self) -> None:
        if not self.lower:
            self.lower = [t.lower() for t in self.tokens]

    def __len__(self) -> int:
        return len(self.tokens)

    @property
    def nonpunct_mask(self) -> np.ndarray:
        return np.array([u != "PUNCT" for u in self.upos], dtype=bool)


# --------------------------------------------------------------------------- download / cache
def _to_str_list(values, feature) -> list[str]:
    """Older loader schemas stored upos as ClassLabel ints; map them back to strings."""
    if values and isinstance(values[0], int):
        names = feature.feature.names  # Sequence(ClassLabel)
        return [names[v] for v in values]
    return [str(v) for v in values]


def download_ewt(out_dir: Path = C.EWT_DIR) -> str:
    """Download EWT from the HF Hub (pinned revision) and write one JSONL file per split.

    Returns the revision that was actually used.
    """
    from datasets import load_dataset  # imported lazily: only needed once

    out_dir.mkdir(parents=True, exist_ok=True)
    last_err = None
    for rev in C.UD_REVISIONS:
        try:
            ds = load_dataset(C.UD_REPO, C.UD_CONFIG, revision=rev)
            break
        except Exception as e:  # noqa: BLE001 - try the next pinned revision
            print(f"[download_ewt] revision {rev!r} failed: {type(e).__name__}: {e}")
            last_err = e
    else:
        raise RuntimeError("Could not load UD English EWT from any pinned revision") from last_err

    print(f"[download_ewt] loaded revision {rev!r}; splits = {list(ds.keys())}")
    for split in C.SPLITS:
        d = ds[split]
        upos_feat = d.features["upos"]
        with open(out_dir / f"{split}.jsonl", "w", encoding="utf-8") as f:
            for ex in d:
                rec = dict(
                    sent_id=ex.get("sent_id", ""),
                    text=ex.get("text", ""),
                    tokens=list(ex["tokens"]),
                    upos=_to_str_list(list(ex["upos"]), upos_feat),
                    xpos=[str(x) if x is not None else "_" for x in ex["xpos"]],
                    heads=[int(h) for h in ex["head"]],
                    deprels=list(ex["deprel"]),
                    n_mwt=len(ex.get("mwt") or []),
                    n_empty=len(ex.get("empty_nodes") or []),
                )
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    (out_dir / "REVISION").write_text(rev + "\n")
    return rev


def load_ewt(split: str, data_dir: Path = C.EWT_DIR) -> list[Sentence]:
    path = data_dir / f"{split}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"{path} missing - run `python code/common/prepare_data.py` first")
    with open(path, encoding="utf-8") as f:
        return [Sentence(**json.loads(line)) for line in f]


# --------------------------------------------------------------------------- tree utilities
def tree_problems(heads: list[int]) -> list[str]:
    """Return a list of problems (empty list == valid single-rooted tree)."""
    n = len(heads)
    problems = []
    roots = [i for i, h in enumerate(heads) if h == 0]
    if len(roots) != 1:
        problems.append(f"{len(roots)} roots")
    if any(h < 0 or h > n for h in heads):
        problems.append("head index out of range")
        return problems
    # cycle check: following heads from every node must reach 0 within n steps
    for i in range(n):
        cur, steps = i + 1, 0
        while cur != 0 and steps <= n:
            cur = heads[cur - 1]
            steps += 1
        if cur != 0:
            problems.append("cycle")
            break
    return problems


def undirected_adjacency(heads: list[int]) -> list[list[int]]:
    adj: list[list[int]] = [[] for _ in heads]
    for i, h in enumerate(heads):
        if h > 0:
            adj[i].append(h - 1)
            adj[h - 1].append(i)
    return adj


def gold_distance_matrix(heads: list[int]) -> np.ndarray:
    """Pairwise path length (number of edges) in the undirected dependency tree. Shape [n, n]."""
    n = len(heads)
    adj = undirected_adjacency(heads)
    dist = np.zeros((n, n), dtype=np.float32)
    for s in range(n):
        seen = [-1] * n
        seen[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if seen[v] < 0:
                    seen[v] = seen[u] + 1
                    q.append(v)
        dist[s] = seen
    return dist


def gold_depths(heads: list[int]) -> np.ndarray:
    """Depth of each word = distance to the root word (root has depth 0). Shape [n]."""
    n = len(heads)
    depth = np.zeros(n, dtype=np.float32)
    for i in range(n):
        d, cur = 0, i + 1
        while heads[cur - 1] != 0:
            cur = heads[cur - 1]
            d += 1
        depth[i] = d
    return depth


def gold_edges(heads: list[int], keep: np.ndarray | None = None) -> set[tuple[int, int]]:
    """Undirected gold edges (i<j, 0-based) among words with keep[i] True (both ends kept)."""
    edges = set()
    for i, h in enumerate(heads):
        if h > 0:
            a, b = sorted((i, h - 1))
            if keep is None or (keep[a] and keep[b]):
                edges.add((a, b))
    return edges


# --------------------------------------------------------------------------- vocabulary helpers
def word_counts(sentences: Iterable[Sentence]) -> Counter:
    c: Counter = Counter()
    for s in sentences:
        c.update(s.lower)
    return c


def nonpunct_words(s: Sentence) -> list[str]:
    return [w for w, u in zip(s.lower, s.upos) if u != "PUNCT"]


def mst_eligible(s: Sentence, vocab: set[str]) -> bool:
    """Decision D10: 5-40 non-punct words, all of them in the (shared T1/T3) train vocab."""
    words = nonpunct_words(s)
    return C.MST_MIN_WORDS <= len(words) <= C.MST_MAX_WORDS and all(w in vocab for w in words)


def sample_mst_sentences(test: list[Sentence], vocab: set[str], k: int = C.MST_SAMPLE_SIZE,
                         seed: int = C.SEED) -> list[str]:
    """Deterministically sample k eligible test sentence ids (decision D10)."""
    eligible = sorted(s.sent_id for s in test if mst_eligible(s, vocab))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(eligible), size=min(k, len(eligible)), replace=False)
    return [eligible[i] for i in sorted(idx)]


def to_dict(s: Sentence) -> dict:
    return asdict(s)


In [ ]:
%%writefile code/common/features.py
"""Access to the cached BERT word features written by task2/extract_bert.py.

Layout (in config.FEATURE_DIR):
    {split}_layer{L}.npy   float16 [n_words_in_split, 768], L = 0..12
                           (L = 0: embedding layer, L = 1..12: transformer layers;
                            = HF `hidden_states[L]`)
    {split}_offsets.npy    int64 [n_sentences + 1]; words of sentence s are rows
                           offsets[s] : offsets[s+1], in the original EWT order
    {split}_meta.json      sent_ids, counts, model + revision, pooling description

Each word vector is the MEAN of its wordpiece vectors (Hewitt & Manning 2019).
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from . import config as C


def layer_path(split: str, layer: int, feat_dir: Path = C.FEATURE_DIR) -> Path:
    return feat_dir / f"{split}_layer{layer}.npy"


def load_offsets(split: str, feat_dir: Path = C.FEATURE_DIR) -> np.ndarray:
    return np.load(feat_dir / f"{split}_offsets.npy")


def load_layer(split: str, layer: int, mmap: bool = True, feat_dir: Path = C.FEATURE_DIR) -> np.ndarray:
    """[n_words, 768] float16. With mmap=True nothing is read until it is indexed."""
    return np.load(layer_path(split, layer, feat_dir), mmap_mode="r" if mmap else None)


def load_meta(split: str, feat_dir: Path = C.FEATURE_DIR) -> dict:
    return json.loads((feat_dir / f"{split}_meta.json").read_text())


def sentence_slices(offsets: np.ndarray) -> list[slice]:
    return [slice(int(a), int(b)) for a, b in zip(offsets[:-1], offsets[1:])]


In [ ]:
%%writefile code/common/prepare_data.py
"""Phase 0: download all data + model and print sanity statistics.

Usage (from the project root, inside the `nlp5` conda env):
    python code/common/prepare_data.py              # everything
    python code/common/prepare_data.py --skip-bert  # no model download / wordpiece stats
    python code/common/prepare_data.py --force      # re-download EWT even if cached

Writes:
    data/ewt/{train,dev,test}.jsonl, data/ewt/REVISION
    data/benchmarks/{wordsim353.tsv, questions-words.txt}
    outputs/shared/data_stats.json
"""
from __future__ import annotations

import argparse
import json
import platform
import sys
from collections import Counter
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))  # make `common` importable

from common import benchmarks as B  # noqa: E402
from common import config as C  # noqa: E402
from common.data import (download_ewt, gold_distance_matrix, gold_edges, load_ewt,  # noqa: E402
                         mst_eligible, tree_problems, word_counts)
from common.trees import gold_mst_edges, mst_edges  # noqa: E402


def versions() -> dict:
    out = {"python": platform.python_version()}
    for mod in ("numpy", "scipy", "sklearn", "pandas", "matplotlib", "torch", "transformers",
                "datasets", "huggingface_hub", "tqdm"):
        try:
            out[mod] = __import__(mod).__version__
        except Exception:  # noqa: BLE001
            out[mod] = "not installed"
    try:
        import torch
        out["cuda_available"] = torch.cuda.is_available()
        out["cuda_devices"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:  # noqa: BLE001
        pass
    return out


def split_stats(sents) -> dict:
    lens = np.array([len(s) for s in sents])
    n_punct = sum(sum(u == "PUNCT" for u in s.upos) for s in sents)
    invalid = [(s.sent_id, tree_problems(s.heads)) for s in sents if tree_problems(s.heads)]
    # self-test 1: MST of the full gold-distance matrix must reproduce the gold tree exactly
    mst_mismatch = sum(mst_edges(gold_distance_matrix(s.heads)) != gold_edges(s.heads) for s in sents)
    # open question: punctuation that has dependents (non-leaf) -> gold MST on non-punct words
    # differs from the gold edges restricted to non-punct words
    punct_nonleaf_tokens = 0
    punct_nonleaf_sents = 0
    for s in sents:
        has_dep = Counter(h - 1 for h in s.heads if h > 0)
        k = sum(1 for i, u in enumerate(s.upos) if u == "PUNCT" and has_dep[i] > 0)
        punct_nonleaf_tokens += k
        punct_nonleaf_sents += k > 0
    return dict(
        n_sentences=len(sents), n_tokens=int(lens.sum()), mean_len=float(lens.mean()),
        median_len=float(np.median(lens)), max_len=int(lens.max()),
        n_len_5_50=int(((lens >= C.SPEARMAN_MIN_LEN) & (lens <= C.SPEARMAN_MAX_LEN)).sum()),
        n_punct_tokens=int(n_punct), sentences_with_mwt=sum(s.n_mwt > 0 for s in sents),
        sentences_with_empty_nodes=sum(s.n_empty > 0 for s in sents),
        n_invalid_trees=len(invalid), invalid_examples=invalid[:5],
        mst_selftest_mismatches=int(mst_mismatch),
        punct_nonleaf_tokens=punct_nonleaf_tokens, punct_nonleaf_sentences=punct_nonleaf_sents,
        example=dict(sent_id=sents[0].sent_id, tokens=sents[0].tokens[:15], upos=sents[0].upos[:15],
                     heads=sents[0].heads[:15], deprels=sents[0].deprels[:15]),
    )


def wordpiece_stats(splits: dict) -> dict:
    from transformers import AutoModel, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(C.BERT_MODEL, revision=C.BERT_REVISION)
    AutoModel.from_pretrained(C.BERT_MODEL, revision=C.BERT_REVISION)  # cache weights only
    out = {}
    for name, sents in splits.items():
        enc = tok([s.tokens for s in sents], is_split_into_words=True, add_special_tokens=False)
        n_wp = np.array([len(ids) for ids in enc["input_ids"]])
        n_words = np.array([len(s) for s in sents])
        # words that the tokenizer maps to ZERO wordpieces (e.g. odd whitespace/control chars)
        zero_wp = 0
        for i, s in enumerate(sents):
            wids = set(w for w in enc.word_ids(i) if w is not None)
            zero_wp += len(s) - len(wids)
        out[name] = dict(max_wordpieces=int(n_wp.max()), mean_wordpieces=float(n_wp.mean()),
                         n_over_limit=int((n_wp > C.BERT_MAX_WORDPIECES).sum()),
                         wordpieces_per_word=float(n_wp.sum() / n_words.sum()),
                         words_with_zero_wordpieces=int(zero_wp))
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--skip-bert", action="store_true")
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args()
    C.ensure_dirs()
    stats: dict = {"versions": versions()}

    # 1. EWT
    if args.force or not all((C.EWT_DIR / f"{s}.jsonl").exists() for s in C.SPLITS):
        stats["ewt_revision"] = download_ewt()
    else:
        stats["ewt_revision"] = (C.EWT_DIR / "REVISION").read_text().strip()
        print(f"[prepare] EWT already cached (revision {stats['ewt_revision']})")
    splits = {s: load_ewt(s) for s in C.SPLITS}
    stats["splits"] = {s: split_stats(v) for s, v in splits.items()}

    # 2. vocabulary (lowercased train word types, decision D3) shared by Task 1 and Task 3
    counts = word_counts(splits["train"])
    vocab = set(counts)
    freq = np.array(list(counts.values()))
    test_tokens = [w for s in splits["test"] for w in s.lower]
    stats["vocab"] = dict(
        n_types=len(vocab), n_tokens=int(freq.sum()), types_count_1=int((freq == 1).sum()),
        types_count_2_4=int(((freq >= 2) & (freq <= 4)).sum()), types_count_ge5=int((freq >= 5).sum()),
        test_token_oov_rate=float(np.mean([w not in vocab for w in test_tokens])),
        top20=counts.most_common(20),
    )

    # 3. benchmarks + coverage
    stats["benchmark_sha256"] = B.download_benchmarks()
    ws = B.load_wordsim()
    an = B.load_analogies()
    stats["wordsim"] = B.wordsim_coverage(ws, vocab)
    cov_words = [w for a, b, _ in ws for w in (a, b) if w in vocab]
    stats["wordsim"]["median_train_count_of_covered_words"] = float(np.median([counts[w] for w in cov_words]))
    stats["wordsim"]["oov_words_sample"] = sorted({w for a, b, _ in ws for w in (a, b) if w not in vocab})[:30]
    stats["analogy"] = B.analogy_coverage(an, vocab)

    # 4. 20-sentence MST sample eligibility (decision D10)
    eligible = [s for s in splits["test"] if mst_eligible(s, vocab)]
    stats["mst_eligible_test_sentences"] = len(eligible)

    # 5. BERT wordpieces
    if not args.skip_bert:
        stats["wordpieces"] = wordpiece_stats(splits)

    out = C.SHARED_OUT / "data_stats.json"
    out.write_text(json.dumps(stats, indent=2, default=str))
    print_report(stats)
    print(f"\n[prepare] full stats written to {out}")


def print_report(st: dict) -> None:
    print("\n================ PHASE 0 SANITY REPORT ================")
    print("versions:", st["versions"])
    print("EWT revision:", st["ewt_revision"])
    for name, s in st["splits"].items():
        print(f"\n[{name}] sentences={s['n_sentences']} tokens={s['n_tokens']} mean_len={s['mean_len']:.1f} "
              f"max_len={s['max_len']} len5-50={s['n_len_5_50']} punct={s['n_punct_tokens']}")
        print(f"   MWT sents={s['sentences_with_mwt']} empty-node sents={s['sentences_with_empty_nodes']} "
              f"invalid trees={s['n_invalid_trees']} MST self-test mismatches={s['mst_selftest_mismatches']} "
              f"non-leaf punct tokens={s['punct_nonleaf_tokens']} (in {s['punct_nonleaf_sentences']} sents)")
    print(f"\nexample sentence: {st['splits']['train']['example']}")
    v = st["vocab"]
    print(f"\n[vocab] types={v['n_types']} tokens={v['n_tokens']} count==1: {v['types_count_1']} "
          f"2-4: {v['types_count_2_4']} >=5: {v['types_count_ge5']}  test-token OOV rate={v['test_token_oov_rate']:.3%}")
    w = st["wordsim"]
    print(f"\n[WordSim-353] pairs={w['n_pairs']} covered={w['n_covered']} ({w['pair_coverage']:.1%}); "
          f"word coverage={w['word_coverage']:.1%}; median train count of covered words="
          f"{w['median_train_count_of_covered_words']}")
    print("   OOV sample:", w["oov_words_sample"])
    a = st["analogy"]
    print(f"\n[Google analogy] questions={a['n_questions']} covered={a['n_covered']} ({a['coverage']:.1%})")
    for sec, d in a["per_section"].items():
        print(f"   {sec:32s} {d['covered']:5d}/{d['n']:5d} ({d['coverage']:.1%})")
    print(f"\n[MST sample] eligible test sentences (5-40 non-punct words, all in vocab): "
          f"{st['mst_eligible_test_sentences']}")
    if "wordpieces" in st:
        for name, d in st["wordpieces"].items():
            print(f"[wordpieces:{name}] {d}")
    print("=======================================================")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/common/static_eval.py
"""One evaluation suite for *static* word vectors, shared by Task 1 (PPMI+SVD) and Task 3
(BERT-pooled). Calling the same function guarantees the "identical protocol" that Task 3
requires.

evaluate_static_vectors() computes:
  1. WordSim-353: Spearman (covered + full, median imputation) with a bootstrap 95% CI
  2. Google analogy: 3CosAdd accuracy (covered + full; per section; semantic vs syntactic)
  3. 20-sentence MST check: cosine-distance MST vs gold tree (UUAS, punctuation excluded),
     with linear-chain and random-tree baselines, bootstrap CI, paired bootstrap vs the chain,
     and a per-dependency-relation recall breakdown
  4. The same MST check on the whole eligible test pool (660 sentences): a more stable estimate
  5. Nearest neighbours of a fixed list of probe words (qualitative)
"""
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from . import benchmarks as B
from . import config as C
from .data import Sentence, gold_edges, load_ewt, mst_eligible, sample_mst_sentences, word_counts
from .stats import bootstrap_spearman, bootstrap_uuas, paired_bootstrap_uuas
from .trees import (cosine_distance_matrix, gold_mst_edges, linear_chain_dist, mst_on_subset,
                    random_tree_edges)

PROBE_WORDS = ["good", "bad", "monday", "dog", "car", "money", "food", "said", "president",
               "google", "happy", "computer", "city", "school", "doctor", "two", "the", "of"]
N_RANDOM_TREES = 100  # random-tree baseline: expected UUAS over this many draws per sentence


# --------------------------------------------------------------------------- the shared 20-sentence sample
def get_mst_sentences() -> tuple[list[Sentence], list[Sentence]]:
    """Return (the 20 sampled test sentences, all eligible test sentences).

    The sample is created once (seed 42, decision D10) and saved to
    outputs/shared/mst_sample_ids.json. Task 3 reloads exactly the same ids.
    """
    train, test = load_ewt("train"), load_ewt("test")
    vocab = set(word_counts(train))  # lowercased train types == the T1/T3 vocabulary
    pool = [s for s in test if mst_eligible(s, vocab)]
    path = C.MST_SAMPLE_PATH
    if path.exists():
        ids = json.loads(path.read_text())["sent_ids"]
    else:
        ids = sample_mst_sentences(test, vocab)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(dict(
            sent_ids=ids, seed=C.SEED, n_eligible=len(pool),
            criteria=f"test split; {C.MST_MIN_WORDS}-{C.MST_MAX_WORDS} non-PUNCT words; "
                     "all non-PUNCT words (lowercased) in the train vocabulary"), indent=2))
        print(f"[static_eval] sampled {len(ids)} MST sentences -> {path}")
    by_id = {s.sent_id: s for s in test}
    return [by_id[i] for i in ids], pool


# --------------------------------------------------------------------------- MST check
def _sentence_vectors(s: Sentence, vectors: np.ndarray, w2i: dict[str, int]) -> np.ndarray:
    """Static vector per token. Non-punct words are guaranteed in-vocab by the sampling;
    OOV punctuation (excluded from the MST anyway) gets a zero vector."""
    out = np.zeros((len(s), vectors.shape[1]), dtype=np.float64)
    for i, w in enumerate(s.lower):
        if w in w2i:
            out[i] = vectors[w2i[w]]
    return out


def mst_check(sentences: list[Sentence], dist_fn, name: str, seed: int = C.SEED,
              n_random: int = N_RANDOM_TREES) -> dict:
    """Generic structural check. dist_fn(sentence) -> [n, n] predicted distance matrix.
    n_random = 0 skips the (slow) random-tree baseline, e.g. inside layer sweeps.

    Returns aggregate UUAS (+CI), the baselines, a paired test vs the chain, per-relation
    recall and per-sentence records (with edges, for the qualitative figures).
    """
    rng = np.random.default_rng(seed)
    per_sent, rel = [], defaultdict(lambda: [0, 0])  # deprel -> [recovered, total]
    for s in sentences:
        keep = s.nonpunct_mask
        gold = gold_mst_edges(s.heads, keep)
        pred = mst_on_subset(dist_fn(s), keep)
        chain = mst_on_subset(linear_chain_dist(len(s)), keep)
        idx = np.flatnonzero(keep)
        rand_correct = np.mean([
            len(gold & {(int(idx[a]), int(idx[b])) for a, b in random_tree_edges(len(idx), rng)})
            for _ in range(n_random)]) if n_random else float("nan")
        for i, h in enumerate(s.heads):  # per-relation recall on the true head edges
            if h > 0 and keep[i] and keep[h - 1]:
                e = tuple(sorted((i, h - 1)))
                r = s.deprels[i].split(":")[0]
                rel[r][1] += 1
                rel[r][0] += e in pred
        per_sent.append(dict(
            sent_id=s.sent_id, text=s.text, n_words=int(keep.sum()), total=len(gold),
            correct=len(gold & pred), chain_correct=len(gold & chain), random_correct=float(rand_correct),
            gold_edges=sorted(gold), pred_edges=sorted(pred), tokens=s.tokens))

    corr = np.array([p["correct"] for p in per_sent])
    chain_c = np.array([p["chain_correct"] for p in per_sent])
    rand_c = np.array([p["random_correct"] for p in per_sent])
    tot = np.array([p["total"] for p in per_sent])
    return dict(
        name=name, n_sentences=len(per_sent), n_edges=int(tot.sum()),
        uuas=float(corr.sum() / tot.sum()), uuas_ci95=bootstrap_uuas(corr, tot),
        chain_uuas=float(chain_c.sum() / tot.sum()), chain_ci95=bootstrap_uuas(chain_c, tot),
        random_uuas=float(rand_c.sum() / tot.sum()),
        vs_chain=paired_bootstrap_uuas(corr, chain_c, tot),
        vs_random=paired_bootstrap_uuas(corr, rand_c, tot) if n_random else None,
        per_relation={k: dict(recovered=v[0], total=v[1], recall=v[0] / v[1])
                      for k, v in sorted(rel.items(), key=lambda kv: -kv[1][1])},
        per_sentence=per_sent,
    )


def static_mst_check(sentences, vectors, w2i, name, n_random: int = N_RANDOM_TREES) -> dict:
    return mst_check(sentences, lambda s: cosine_distance_matrix(_sentence_vectors(s, vectors, w2i)), name,
                     n_random=n_random)


def per_sentence_frame(m: dict) -> pd.DataFrame:
    """Compact per-sentence counts (for paired bootstrap tests between methods)."""
    return pd.DataFrame([dict(sent_id=p["sent_id"], correct=p["correct"], total=p["total"],
                              chain_correct=p["chain_correct"]) for p in m["per_sentence"]])


# --------------------------------------------------------------------------- qualitative
def nearest_neighbours(vectors: np.ndarray, w2i: dict[str, int], words=PROBE_WORDS, k: int = 10) -> dict:
    U = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    i2w = {i: w for w, i in w2i.items()}
    out = {}
    for w in words:
        if w not in w2i:
            continue
        sims = U @ U[w2i[w]]
        sims[w2i[w]] = -np.inf
        top = np.argsort(-sims)[:k]
        out[w] = [(i2w[int(j)], round(float(sims[j]), 3)) for j in top]
    return out


# --------------------------------------------------------------------------- full suite
def evaluate_static_vectors(vectors: np.ndarray, w2i: dict[str, int], name: str, out_dir: Path,
                            run_pool: bool = True, verbose: bool = True) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    vectors = np.asarray(vectors, dtype=np.float32)

    # 1. WordSim-353
    ws = B.evaluate_wordsim(vectors, w2i)
    recs = pd.DataFrame(ws.pop("records"))
    recs.to_csv(out_dir / "wordsim_pairs.csv", index=False)
    cov = recs[recs.covered]
    ws["spearman_covered_ci95"] = bootstrap_spearman(cov.model.values, cov.human.values)
    # frequency-stratified Spearman: are low scores caused by rare (poorly estimated) words?
    counts = word_counts(load_ewt("train"))
    minf = cov.apply(lambda r: min(counts[r.word1], counts[r.word2]), axis=1)
    ws["by_min_train_freq"] = {
        str(t): dict(n=int((minf >= t).sum()),
                     spearman=float(spearmanr(cov.model[minf >= t], cov.human[minf >= t]).statistic)
                     if (minf >= t).sum() >= 10 else float("nan"))
        for t in (1, 5, 10, 20, 50)}

    # 2. analogies
    an = B.evaluate_analogy(vectors, w2i)
    pd.DataFrame(an.pop("examples")).to_csv(out_dir / "analogy_examples.csv", index=False)
    pd.DataFrame([dict(section=k, **v) for k, v in an["per_section"].items()]).to_csv(
        out_dir / "analogy_by_section.csv", index=False)

    # 3 + 4. structural check
    sample, pool = get_mst_sentences()
    mst20 = static_mst_check(sample, vectors, w2i, name)
    (out_dir / "mst_20_sentences.json").write_text(json.dumps(mst20, indent=1))
    mst20_summary = {k: v for k, v in mst20.items() if k not in ("per_sentence",)}
    pool_summary = None
    if run_pool:
        mpool = static_mst_check(pool, vectors, w2i, name)
        per_sentence_frame(mpool).to_csv(out_dir / "mst_pool_per_sentence.csv", index=False)
        pool_summary = {k: v for k, v in mpool.items() if k not in ("per_sentence", "per_relation")}
        pool_summary["per_relation"] = mpool["per_relation"]

    # 5. neighbours
    nn = nearest_neighbours(vectors, w2i)
    (out_dir / "nearest_neighbours.json").write_text(json.dumps(nn, indent=1))

    results = dict(name=name, vocab_size=len(w2i), dim=int(vectors.shape[1]), wordsim=ws,
                   analogy=an, mst_20=mst20_summary, mst_pool=pool_summary)
    (out_dir / "results.json").write_text(json.dumps(results, indent=2))
    if verbose:
        print_summary(results, nn)
    return results


def print_summary(r: dict, nn: dict | None = None) -> None:
    ws, an, m = r["wordsim"], r["analogy"], r["mst_20"]
    lo, hi = ws["spearman_covered_ci95"]
    print(f"\n================ STATIC-VECTOR EVALUATION: {r['name']} ================")
    print(f"vocab={r['vocab_size']} dim={r['dim']}")
    print(f"[WordSim-353] Spearman covered = {ws['spearman_covered']:.4f}  (95% CI {lo:.3f}..{hi:.3f}, "
          f"n={ws['n_covered']}/{ws['n_total']})   full (median-imputed) = {ws['spearman_full']:.4f}")
    if "by_min_train_freq" in ws:
        print("   by min train freq of the pair: " + ", ".join(
            f">={t}: {v['spearman']:.3f} (n={v['n']})" for t, v in ws["by_min_train_freq"].items()))
    print(f"[Analogy] acc covered = {an['acc_covered']:.4f} ({an['n_correct']}/{an['n_covered']})   "
          f"acc full = {an['acc_full']:.4f} (/{an['n_total']})")
    for k, v in an["by_kind"].items():
        print(f"   {k:9s}: covered acc = {v['acc_covered']:.4f} ({v['correct']}/{v['covered']})  full = {v['acc_full']:.4f}")
    for sec, v in an["per_section"].items():
        print(f"   {sec:30s} {v['correct']:5d}/{v['covered']:5d} = {v['acc_covered']:.3f}")
    lo, hi = m["uuas_ci95"]
    print(f"[MST 20 sents] UUAS = {m['uuas']:.4f} (95% CI {lo:.3f}..{hi:.3f}; {m['n_edges']} gold edges)  "
          f"| linear chain = {m['chain_uuas']:.4f} | random tree = {m['random_uuas']:.4f}")
    print(f"   vs chain: diff={m['vs_chain']['diff']:+.4f} CI [{m['vs_chain']['ci_low']:+.3f},{m['vs_chain']['ci_high']:+.3f}]"
          f"   vs random: diff={m['vs_random']['diff']:+.4f} CI [{m['vs_random']['ci_low']:+.3f},{m['vs_random']['ci_high']:+.3f}]")
    if r.get("mst_pool"):
        p = r["mst_pool"]
        print(f"[MST eligible pool, {p['n_sentences']} sents] UUAS = {p['uuas']:.4f} "
              f"(CI {p['uuas_ci95'][0]:.3f}..{p['uuas_ci95'][1]:.3f}) | chain = {p['chain_uuas']:.4f} | random = {p['random_uuas']:.4f}")
        top = list(p["per_relation"].items())[:12]
        print("   recall by relation (pool): " + ", ".join(f"{k}={v['recall']:.2f}(n={v['total']})" for k, v in top))
    if nn:
        print("[nearest neighbours]")
        for w, lst in nn.items():
            print(f"   {w:10s} -> " + ", ".join(x for x, _ in lst[:8]))
    print("=" * 72)


In [ ]:
%%writefile code/common/stats.py
"""Bootstrap confidence intervals (decision D13). Seeded, so results are reproducible."""
from __future__ import annotations

import numpy as np
from scipy.stats import spearmanr

from . import config as C


def _ci(samples: np.ndarray, alpha: float = 0.05) -> tuple[float, float]:
    lo, hi = np.nanpercentile(samples, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def bootstrap_spearman(x, y, n: int = C.BOOTSTRAP_RESAMPLES, seed: int = C.SEED) -> tuple[float, float]:
    """95% CI of Spearman rho, resampling (x, y) pairs with replacement."""
    x, y = np.asarray(x), np.asarray(y)
    rng = np.random.default_rng(seed)
    stats = np.empty(n)
    for b in range(n):
        idx = rng.integers(0, len(x), len(x))
        stats[b] = spearmanr(x[idx], y[idx]).statistic
    return _ci(stats)


def bootstrap_uuas(correct, total, n: int = C.BOOTSTRAP_RESAMPLES, seed: int = C.SEED) -> tuple[float, float]:
    """95% CI of micro-averaged UUAS, resampling *sentences* with replacement."""
    correct, total = np.asarray(correct, float), np.asarray(total, float)
    rng = np.random.default_rng(seed)
    stats = np.empty(n)
    for b in range(n):
        idx = rng.integers(0, len(correct), len(correct))
        stats[b] = correct[idx].sum() / max(total[idx].sum(), 1)
    return _ci(stats)


def paired_bootstrap_uuas(correct_a, correct_b, total, n: int = C.BOOTSTRAP_RESAMPLES,
                          seed: int = C.SEED) -> dict:
    """Paired bootstrap for UUAS(A) - UUAS(B) on the same sentences.

    Returns the observed difference, its 95% CI, and the fraction of resamples where A <= B
    (a one-sided p-value style quantity).
    """
    a, b, t = (np.asarray(v, float) for v in (correct_a, correct_b, total))
    rng = np.random.default_rng(seed)
    diffs = np.empty(n)
    for k in range(n):
        idx = rng.integers(0, len(t), len(t))
        diffs[k] = (a[idx].sum() - b[idx].sum()) / max(t[idx].sum(), 1)
    obs = (a.sum() - b.sum()) / t.sum()
    lo, hi = _ci(diffs)
    return dict(diff=float(obs), ci_low=lo, ci_high=hi, frac_le_zero=float((diffs <= 0).mean()))


def paired_bootstrap_spearman(model_a, model_b, human, n: int = C.BOOTSTRAP_RESAMPLES,
                              seed: int = C.SEED) -> dict:
    """Paired bootstrap for rho(A, human) - rho(B, human) on the *same* word pairs."""
    a, b, h = (np.asarray(v, float) for v in (model_a, model_b, human))
    rng = np.random.default_rng(seed)
    diffs = np.empty(n)
    for k in range(n):
        idx = rng.integers(0, len(h), len(h))
        diffs[k] = spearmanr(a[idx], h[idx]).statistic - spearmanr(b[idx], h[idx]).statistic
    obs = spearmanr(a, h).statistic - spearmanr(b, h).statistic
    lo, hi = _ci(diffs)
    return dict(diff=float(obs), ci_low=lo, ci_high=hi, frac_le_zero=float((diffs <= 0).mean()))


def mcnemar_exact(correct_a, correct_b) -> dict:
    """Exact (binomial) McNemar test for two classifiers on the same items (e.g. analogy questions)."""
    from scipy.stats import binomtest
    a, b = np.asarray(correct_a, bool), np.asarray(correct_b, bool)
    a_only, b_only = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return dict(a_only=a_only, b_only=b_only, p_value=float(p))


In [ ]:
%%writefile code/common/trees.py
"""Minimum spanning trees, UUAS and simple tree baselines (shared by T1, T2, T3).

UUAS follows Hewitt & Manning's reporter.py: an MST is built over the *predicted* distance
matrix and over the *gold* tree-distance matrix, both restricted to non-punctuation words,
and the undirected edge sets are compared. Gold and predicted trees both have m-1 edges,
so precision = recall = UUAS.
"""
from __future__ import annotations

import numpy as np

from .data import gold_distance_matrix

Edge = tuple[int, int]


def mst_edges(dist: np.ndarray) -> set[Edge]:
    """Prim's algorithm on a dense symmetric distance matrix. Returns undirected edges (i<j).

    Deterministic: starts at node 0, and ties are broken by the lowest index (np.argmin).
    """
    n = dist.shape[0]
    if n <= 1:
        return set()
    in_tree = np.zeros(n, dtype=bool)
    in_tree[0] = True
    best = dist[0].astype(np.float64).copy()
    parent = np.zeros(n, dtype=int)
    edges = set()
    for _ in range(n - 1):
        cand = np.where(in_tree, np.inf, best)
        v = int(np.argmin(cand))
        edges.add(tuple(sorted((int(parent[v]), v))))
        in_tree[v] = True
        closer = dist[v] < best
        best = np.where(closer, dist[v], best)
        parent = np.where(closer, v, parent)
    return edges


def _restrict(dist: np.ndarray, keep: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    idx = np.flatnonzero(keep)
    return dist[np.ix_(idx, idx)], idx


def mst_on_subset(dist: np.ndarray, keep: np.ndarray) -> set[Edge]:
    """MST over the words with keep=True, returned in ORIGINAL word indices."""
    sub, idx = _restrict(dist, keep)
    return {(int(idx[a]), int(idx[b])) for a, b in mst_edges(sub)}


def gold_mst_edges(heads: list[int], keep: np.ndarray) -> set[Edge]:
    """Gold tree among kept words: MST of the gold distance matrix (equals the gold edges
    whenever the removed punctuation tokens are leaves, which is almost always the case)."""
    return mst_on_subset(gold_distance_matrix(heads), keep)


def uuas_counts(pred_dist: np.ndarray, heads: list[int], keep: np.ndarray) -> tuple[int, int]:
    """(# correct undirected edges, # gold edges) for one sentence."""
    gold = gold_mst_edges(heads, keep)
    pred = mst_on_subset(pred_dist, keep)
    return len(gold & pred), len(gold)


def cosine_distance_matrix(vecs: np.ndarray) -> np.ndarray:
    v = np.asarray(vecs, dtype=np.float64)
    v = v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-12)
    return 1.0 - v @ v.T


# --------------------------------------------------------------------------- baselines
def linear_chain_dist(n: int) -> np.ndarray:
    """Distance |i-j|: its MST is the chain w1-w2-...-wn (Hewitt's LINEAR baseline)."""
    r = np.arange(n)
    return np.abs(r[:, None] - r[None, :]).astype(np.float64)


def random_tree_edges(n: int, rng: np.random.Generator) -> set[Edge]:
    """Uniformly random labelled tree on n nodes via a random Pruefer sequence."""
    if n <= 1:
        return set()
    if n == 2:
        return {(0, 1)}
    seq = rng.integers(0, n, size=n - 2).tolist()
    degree = [1] * n
    for x in seq:
        degree[x] += 1
    edges = set()
    for x in seq:
        leaf = next(i for i in range(n) if degree[i] == 1)
        edges.add(tuple(sorted((leaf, x))))
        degree[leaf] -= 1
        degree[x] -= 1
    u, v = [i for i in range(n) if degree[i] == 1]
    edges.add((u, v))
    return edges


def random_tree_uuas_counts(heads: list[int], keep: np.ndarray, rng: np.random.Generator) -> tuple[int, int]:
    gold = gold_mst_edges(heads, keep)
    idx = np.flatnonzero(keep)
    pred = {(int(idx[a]), int(idx[b])) for a, b in random_tree_edges(len(idx), rng)}
    return len(gold & pred), len(gold)


In [ ]:
%%writefile code/run_all.py
"""Run the whole project (or one task) end to end, in the right order.

Usage (from the project root):
    python code/run_all.py                 # data + Task 1 + Task 2 + Task 3 (full; ~2-4 h on one GPU)
    python code/run_all.py --tasks 1       # one task (its inputs must exist: Task 3 needs Task 2's outputs)
    python code/run_all.py --tasks 2 3 --fast
    python code/run_all.py --dry-run       # print the commands only

Steps (each is an ordinary script that can also be run on its own; see README.md):
  data   common/prepare_data.py           EWT + WS-353 + Google analogies + BERT cache, sanity report
  task1  task1/build_ppmi_svd.py + evaluate_task1.py (+ Levy et al. ablations, summary table)
  task2  task2/extract_bert.py -> train_probes.py -> evaluate_task2.py -> plot_task2.py -> layer_significance.py
  task3  task3/pool_static.py -> evaluate_task3.py -> extensions_task3.py -> diagnostics_task3.py -> plot_task3.py

--fast trains the probes for rank 64 only (26 instead of 78 probes); every other step is identical.
--with-control also runs the spec-literal Task 2 control (constant lr, last epoch, k = 128).
Steps whose outputs already exist are skipped by the scripts themselves where that is cheap to check
(probe checkpoints); re-running is otherwise safe (outputs are overwritten deterministically, seed 42).
"""
from __future__ import annotations

import argparse
import shlex
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
PY = sys.executable

T1_ABLATIONS = {  # tag -> extra flags (Levy, Goldberg & Dagan 2015); "p0" is used by Task 3 as the best T1 variant
    "p0": ["--eig-power", "0"],
    "p05": ["--eig-power", "0.5"],
    "cds075": ["--cds-alpha", "0.75"],
    "cds075_p05": ["--cds-alpha", "0.75", "--eig-power", "0.5"],
}


def steps(args) -> list[tuple[str, list[str]]]:
    s: list[tuple[str, list[str]]] = []
    if "data" in args.tasks:
        s.append(("data", ["code/common/prepare_data.py"]))
    if "1" in args.tasks:
        s.append(("task1", ["code/task1/build_ppmi_svd.py"]))
        s.append(("task1", ["code/task1/evaluate_task1.py"]))
        for tag, flags in T1_ABLATIONS.items():
            if args.skip_ablations and tag != "p0":
                continue
            s.append(("task1", ["code/task1/build_ppmi_svd.py", *flags, "--tag", tag]))
            s.append(("task1", ["code/task1/evaluate_task1.py", "--tag", tag]))
        s.append(("task1", ["code/task1/summarize_task1.py"]))
    if "2" in args.tasks:
        s.append(("task2", ["code/task2/extract_bert.py"]))
        s.append(("task2", ["code/task2/train_probes.py", *(["--ranks", "64"] if args.fast else [])]))
        s.append(("task2", ["code/task2/evaluate_task2.py"]))
        s.append(("task2", ["code/task2/plot_task2.py"]))
        s.append(("task2", ["code/task2/layer_significance.py"]))
        if args.with_control:
            s.append(("task2", ["code/task2/train_probes.py", "--ranks", "128", "--no-decay", "--keep-last",
                                "--tag", "spec_literal"]))
            s.append(("task2", ["code/task2/evaluate_task2.py", "--probe-dir", "outputs/task2/probes_spec_literal",
                                "--out-dir", "outputs/task2/spec_literal"]))
    if "3" in args.tasks:
        s.append(("task3", ["code/task3/pool_static.py"]))
        s.append(("task3", ["code/task3/evaluate_task3.py"]))
        s.append(("task3", ["code/task3/extensions_task3.py"]))
        s.append(("task3", ["code/task3/diagnostics_task3.py"]))
        s.append(("task3", ["code/task3/plot_task3.py"]))
    return s


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tasks", nargs="+", default=["data", "1", "2", "3"], choices=["data", "1", "2", "3"])
    ap.add_argument("--fast", action="store_true", help="Task 2: rank 64 only (26 probes)")
    ap.add_argument("--skip-ablations", action="store_true", help="Task 1: only the p0 variant (needed by Task 3)")
    ap.add_argument("--with-control", action="store_true", help="Task 2: also the spec-literal control run")
    ap.add_argument("--dry-run", action="store_true")
    args = ap.parse_args()

    plan = steps(args)
    log_dir = ROOT / "outputs" / "run_all_logs"
    t_all = time.time()
    for i, (task, cmd) in enumerate(plan, 1):
        line = " ".join(shlex.quote(c) for c in [Path(PY).name, *cmd])
        print(f"\n[{i}/{len(plan)}] ({task}) {line}", flush=True)
        if args.dry_run:
            continue
        log_dir.mkdir(parents=True, exist_ok=True)
        log = log_dir / f"{i:02d}_{Path(cmd[0]).stem}{'_' + cmd[-1] if '--tag' in cmd else ''}.log"
        t0 = time.time()
        with open(log, "w") as fh:
            proc = subprocess.Popen([PY, *cmd], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for out_line in proc.stdout:  # stream to the console and to the log
                sys.stdout.write(out_line)
                fh.write(out_line)
            rc = proc.wait()
        print(f"    -> exit {rc} in {time.time() - t0:.0f} s (log: {log.relative_to(ROOT)})", flush=True)
        if rc != 0:
            sys.exit(f"[run_all] step {i} failed; fix the problem and re-run (finished probes are kept).")
    if not args.dry_run:
        print(f"\n[run_all] all {len(plan)} steps done in {(time.time() - t_all) / 60:.1f} min. "
              f"Results: outputs/task1, outputs/task2, outputs/task3 (see README.md).")


if __name__ == "__main__":
    main()


### Data download and sanity report

In [ ]:
run("code/run_all.py", "--tasks", "data")

## 3. Task 1 — PPMI + SVD baseline (`code/task1/`)
Co-occurrence (±5 words) → PPMI → truncated SVD (100 d); evaluated on WordSim-353, Google analogies and the
20-sentence MST check, plus the Levy et al. (2015) variants.

In [ ]:
%%writefile code/task1/build_ppmi_svd.py
"""Task 1, steps 1-3: co-occurrence matrix -> PPMI -> truncated SVD (100-d static vectors).

Usage (from the project root):
    python code/task1/build_ppmi_svd.py                      # spec setting (writes outputs/task1/)
    python code/task1/build_ppmi_svd.py --cds-alpha 0.75 --eig-power 0.5 --tag cds075_p05
                                                             # (Tier C) ablation -> outputs/task1/ablations/<tag>/

Spec setting: EWT train split, raw words only (lowercased, decision D3; punctuation tokens
are kept as ordinary words), symmetric window +-5 inside sentence boundaries, unweighted
counts, plain PPMI, TruncatedSVD(100, random_state=42), vectors = U * Sigma.
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.data import load_ewt  # noqa: E402


def build_vocab(sentences, min_count: int) -> tuple[list[str], Counter]:
    counts = Counter(w for s in sentences for w in s.lower)
    # deterministic order: frequency desc, then alphabetical
    vocab = sorted((w for w, c in counts.items() if c >= min_count), key=lambda w: (-counts[w], w))
    return vocab, counts


def cooccurrence_matrix(sentences, w2i: dict[str, int], window: int) -> sp.csr_matrix:
    """Symmetric word-word counts: C[w, c] = # times c occurs within +-window of w (same sentence).

    Vectorised: concatenate all sentences into one id stream with a parallel sentence-id stream,
    and for each offset d = 1..window pair position k with k+d when both are in the same sentence.
    Out-of-vocab words (only when min_count > 1) are dropped before windowing, as in word2vec.
    """
    ids, sid = [], []
    for n, s in enumerate(sentences):
        toks = [w2i[w] for w in s.lower if w in w2i]
        ids.extend(toks)
        sid.extend([n] * len(toks))
    ids, sid = np.asarray(ids, dtype=np.int64), np.asarray(sid, dtype=np.int64)
    V = len(w2i)
    rows, cols = [], []
    for d in range(1, window + 1):
        same = sid[:-d] == sid[d:]
        a, b = ids[:-d][same], ids[d:][same]
        rows += [a, b]  # both directions -> symmetric
        cols += [b, a]
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    M = sp.coo_matrix((np.ones_like(rows, dtype=np.float64), (rows, cols)), shape=(V, V)).tocsr()
    M.sum_duplicates()
    return M


def ppmi_matrix(M: sp.csr_matrix, cds_alpha: float = 1.0) -> sp.csr_matrix:
    """PPMI(w, c) = max(0, log [P(w, c) / (P(w) P_alpha(c))]).

    cds_alpha = 1 is plain PPMI (spec). cds_alpha = 0.75 is the context-distribution smoothing
    of Levy, Goldberg & Dagan (2015), only used in the Tier C ablation.
    Computed on the non-zero entries only (zero counts give PMI = -inf, hence PPMI = 0).
    """
    M = M.tocoo()
    total = M.sum()
    row = np.asarray(M.sum(axis=1)).ravel()
    col = np.asarray(M.sum(axis=0)).ravel()
    p_w = row / total
    col_a = col ** cds_alpha
    p_c = col_a / col_a.sum()
    pmi = np.log((M.data / total) / (p_w[M.row] * p_c[M.col]))
    keep = pmi > 0
    P = sp.coo_matrix((pmi[keep], (M.row[keep], M.col[keep])), shape=M.shape).tocsr()
    return P


def svd_vectors(P: sp.csr_matrix, dim: int, eig_power: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Truncated SVD P ~ U S V^T; word vectors = U * S^eig_power (spec: eig_power = 1)."""
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=10, random_state=seed)
    US = svd.fit_transform(P)  # = U * S
    S = svd.singular_values_
    U = US / S
    return (U * S ** eig_power).astype(np.float32), S


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--window", type=int, default=C.WINDOW)
    ap.add_argument("--dim", type=int, default=C.SVD_DIM)
    ap.add_argument("--min-count", type=int, default=C.MIN_COUNT)
    ap.add_argument("--cds-alpha", type=float, default=1.0)
    ap.add_argument("--eig-power", type=float, default=1.0)
    ap.add_argument("--tag", default=None, help="ablation name -> outputs/task1/ablations/<tag>/")
    args = ap.parse_args()
    C.set_seed()
    out = C.TASK1_OUT / "ablations" / args.tag if args.tag else C.TASK1_OUT
    out.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    train = load_ewt("train")
    vocab, counts = build_vocab(train, args.min_count)
    w2i = {w: i for i, w in enumerate(vocab)}
    print(f"[task1] {len(train)} train sentences, vocab = {len(vocab)} types (min_count={args.min_count})")

    M = cooccurrence_matrix(train, w2i, args.window)
    print(f"[task1] co-occurrence: shape={M.shape} nnz={M.nnz} total count={M.sum():.0f} "
          f"symmetric={abs(M - M.T).sum() == 0}")
    P = ppmi_matrix(M, args.cds_alpha)
    print(f"[task1] PPMI: nnz={P.nnz} ({P.nnz / M.nnz:.1%} of co-occurring pairs have PMI > 0), "
          f"max={P.max():.2f}")
    vecs, S = svd_vectors(P, args.dim, args.eig_power, C.SEED)
    zero_rows = int((np.linalg.norm(vecs, axis=1) == 0).sum())
    print(f"[task1] SVD: vectors {vecs.shape}; top-5 singular values {np.round(S[:5], 2).tolist()}; "
          f"zero vectors = {zero_rows}")

    np.save(out / "vectors.npy", vecs)
    (out / "vocab.json").write_text(json.dumps(vocab))
    meta = dict(window=args.window, dim=args.dim, min_count=args.min_count, cds_alpha=args.cds_alpha,
                eig_power=args.eig_power, seed=C.SEED, vocab_size=len(vocab),
                cooc_nnz=int(M.nnz), cooc_total=float(M.sum()), ppmi_nnz=int(P.nnz),
                singular_values=S.tolist(), seconds=round(time.time() - t0, 1),
                word_counts={w: counts[w] for w in vocab})
    (out / "build_meta.json").write_text(json.dumps(meta))
    print(f"[task1] saved vectors + vocab to {out}  ({meta['seconds']} s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task1/evaluate_task1.py
"""Task 1, steps 4-5: evaluate the PPMI+SVD vectors.

Usage (from the project root, after build_ppmi_svd.py):
    python code/task1/evaluate_task1.py                    # spec vectors in outputs/task1/
    python code/task1/evaluate_task1.py --tag <ablation>   # vectors in outputs/task1/ablations/<tag>/

Writes to the same folder: results.json, wordsim_pairs.csv, analogy_by_section.csv,
analogy_examples.csv, mst_20_sentences.json, nearest_neighbours.json.
The first run also creates outputs/shared/mst_sample_ids.json (the 20 sentences reused by Task 3).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.static_eval import evaluate_static_vectors  # noqa: E402


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--tag", default=None)
    ap.add_argument("--no-pool", action="store_true", help="skip the 660-sentence pool MST check")
    args = ap.parse_args()
    C.set_seed()
    d = C.TASK1_OUT / "ablations" / args.tag if args.tag else C.TASK1_OUT
    vecs = np.load(d / "vectors.npy")
    vocab = json.loads((d / "vocab.json").read_text())
    w2i = {w: i for i, w in enumerate(vocab)}
    name = f"T1 PPMI+SVD{'' if not args.tag else ' [' + args.tag + ']'}"
    evaluate_static_vectors(vecs, w2i, name, d, run_pool=not args.no_pool)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task1/summarize_task1.py
"""Collect Task 1 results (spec run + any ablations) into one table.

Usage: python code/task1/summarize_task1.py   ->  outputs/task1/summary_table.csv (+ printed)
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402


def row(tag: str, d: Path) -> dict | None:
    if not (d / "results.json").exists():
        return None
    r = json.loads((d / "results.json").read_text())
    m = json.loads((d / "build_meta.json").read_text())
    ws, an, m20, mp = r["wordsim"], r["analogy"], r["mst_20"], r.get("mst_pool") or {}
    return dict(
        run=tag, cds_alpha=m["cds_alpha"], eig_power=m["eig_power"], min_count=m["min_count"],
        ws_rho_cov=ws["spearman_covered"], ws_ci=f"{ws['spearman_covered_ci95'][0]:.3f}..{ws['spearman_covered_ci95'][1]:.3f}",
        ws_rho_full=ws["spearman_full"], ws_rho_freq20=ws.get("by_min_train_freq", {}).get("20", {}).get("spearman"),
        an_acc_cov=an["acc_covered"], an_sem_cov=an["by_kind"]["semantic"]["acc_covered"],
        an_syn_cov=an["by_kind"]["syntactic"]["acc_covered"], an_acc_full=an["acc_full"],
        mst20_uuas=m20["uuas"], mst20_chain=m20["chain_uuas"], mst20_random=m20["random_uuas"],
        pool_uuas=mp.get("uuas"))


def main() -> None:
    rows = [row("spec (alpha=1, p=1)", C.TASK1_OUT)]
    abl = C.TASK1_OUT / "ablations"
    if abl.exists():
        rows += [row(d.name, d) for d in sorted(abl.iterdir()) if d.is_dir()]
    df = pd.DataFrame([r for r in rows if r])
    df.to_csv(C.TASK1_OUT / "summary_table.csv", index=False)
    with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 4):
        print(df)


if __name__ == "__main__":
    main()


In [ ]:
run("code/run_all.py", "--tasks", "1")

## 4. Task 2 — Structural probe on BERT (`code/task2/`)
BERT features for all 13 layers (cached once) → distance and depth probes for every layer × rank {32, 64, 128}
(Adam 1e-3, batch 20, 30 epochs, ×0.1 lr decay on dev plateau as in the paper) → UUAS / depth Spearman per layer → ℓ*.

In [ ]:
%%writefile code/task2/evaluate_task2.py
"""Task 2: evaluate every trained probe on dev and test, pick the best layer l*, compare to the paper.

Usage (from the project root, after train_probes.py):
    CUDA_VISIBLE_DEVICES=0 python code/task2/evaluate_task2.py

Metrics (exact definitions of Hewitt's structural-probes/reporter.py, checked 6 Nov 2026):
  UUAS   (distance probe) Prim MST on predicted squared distances vs MST on gold distances, both over
         NON-punctuation words; micro-average of correct undirected edges over all sentences.
  DSpr.  (distance probe) Spearman of each word's predicted-distance row vs its gold row (all words incl.
         punctuation and the diagonal); pooled per sentence length; macro-average over lengths 5..50.
  NSpr.  (depth probe; the assignment's "depth-probe Spearman") Spearman of predicted vs gold depths per
         sentence; averaged per length; macro-average over lengths 5..50.
  Root%  (depth probe) argmin of predicted depth over non-punctuation words == gold root.
Baseline: LINEAR from the paper (distance = |i-j|, depth = position i).

l* selection (decision D6): for each layer 1..12 take the rank with the best DEV UUAS; l* = layer with
the best dev UUAS (ties -> higher dev NSpr). Test numbers are reported for every layer and rank.

Outputs (outputs/task2/):
  results_task2.csv          one row per (layer, rank, probe, split)
  per_relation_test.csv      distance-probe recall per dependency relation (test)
  best_layer.json            l*, chosen ranks, dev/test metrics, agreement with the test argmax
  predictions_best.pkl       test predictions of the l* probes and the layer-0 probes (for plots)
  paper_comparison.csv       our l* (and best-test) rows next to Hewitt & Manning Table 1
"""
from __future__ import annotations

import argparse
import json
import pickle
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.trees import gold_mst_edges, linear_chain_dist, mst_on_subset  # noqa: E402

from probe_data import GoldData, LayerFeatures  # noqa: E402
from probes import PROBES  # noqa: E402

PAPER = {  # Hewitt & Manning 2019, Table 1 (PTB WSJ test, Stanford Dependencies)
    "LINEAR": dict(uuas=48.9, dspr=0.58, root=2.9, nspr=0.27),
    "BERTBASE7": dict(uuas=79.8, dspr=0.85, root=88.0, nspr=0.87),
    "BERTLARGE15": dict(uuas=82.5, dspr=0.86, root=89.4, nspr=0.88),
    "BERTLARGE16": dict(uuas=81.7, dspr=0.87, root=90.1, nspr=0.89),
}


# --------------------------------------------------------------------------- metric helpers
def _row_spearman(P: np.ndarray, G: np.ndarray) -> np.ndarray:
    """Spearman per row (average ranks for ties, as scipy.spearmanr). P, G: [m, n]."""
    rp, rg = rankdata(P, axis=1), rankdata(G, axis=1)
    rp -= rp.mean(1, keepdims=True)
    rg -= rg.mean(1, keepdims=True)
    den = np.sqrt((rp ** 2).sum(1) * (rg ** 2).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return (rp * rg).sum(1) / den


def _macro_by_length(per_len: dict[int, list[float]]) -> float:
    means = [np.nanmean(per_len[n]) for n in range(C.SPEARMAN_MIN_LEN, C.SPEARMAN_MAX_LEN + 1) if per_len.get(n)]
    return float(np.mean(means))


class SplitEval:
    """Gold structures of one split, cached once and reused for every probe."""

    def __init__(self, gold: GoldData):
        self.gold = gold
        self.sents = gold.sents
        self.keep = [s.nonpunct_mask for s in self.sents]
        self.gold_dist = [d.cpu().numpy() for d in gold.dist]
        self.gold_depth = [d.cpu().numpy() for d in gold.depth]
        self.gold_edges = [gold_mst_edges(s.heads, k) for s, k in zip(self.sents, self.keep)]
        self.root = [s.heads.index(0) for s in self.sents]

    def distance_metrics(self, preds: list[np.ndarray], with_relations: bool = False) -> dict:
        correct = total = 0
        per_len = defaultdict(list)
        rel = defaultdict(lambda: [0, 0])
        per_sent = []
        for s, P, G, keep, gold in zip(self.sents, preds, self.gold_dist, self.keep, self.gold_edges):
            pred_edges = mst_on_subset(P, keep)
            c = len(gold & pred_edges)
            correct += c
            total += len(gold)
            per_sent.append((c, len(gold)))
            n = len(s)
            if C.SPEARMAN_MIN_LEN <= n <= C.SPEARMAN_MAX_LEN:
                per_len[n].extend(_row_spearman(P, G).tolist())
            if with_relations:
                for i, h in enumerate(s.heads):
                    if h > 0 and keep[i] and keep[h - 1]:
                        r = s.deprels[i].split(":")[0]
                        rel[r][1] += 1
                        rel[r][0] += tuple(sorted((i, h - 1))) in pred_edges
        out = dict(uuas=correct / total, dspr=_macro_by_length(per_len), per_sentence=per_sent)
        if with_relations:
            out["relations"] = {k: dict(recovered=v[0], total=v[1], recall=v[0] / v[1]) for k, v in rel.items()}
        return out

    def depth_metrics(self, preds: list[np.ndarray]) -> dict:
        per_len = defaultdict(list)
        root_ok = 0
        for s, p, g, keep, root in zip(self.sents, preds, self.gold_depth, self.keep, self.root):
            cand = np.where(keep, p, np.inf) if keep.any() else p
            root_ok += int(np.argmin(cand) == root)
            n = len(s)
            if C.SPEARMAN_MIN_LEN <= n <= C.SPEARMAN_MAX_LEN:
                per_len[n].append(float(_row_spearman(p[None], g[None])[0]))
        return dict(nspr=_macro_by_length(per_len), root=root_ok / len(self.sents))


@torch.no_grad()
def predict(model, feats: LayerFeatures, gold: GoldData) -> list[np.ndarray]:
    out = []
    idx_all = torch.arange(len(gold), device=gold.device)
    for s in range(0, len(gold), 100):
        idx = idx_all[s:s + 100]
        pred = model(feats.inputs(idx)).cpu().numpy()
        for b, i in enumerate(idx.tolist()):
            n = gold.lengths[i]
            out.append(pred[b, :n, :n] if pred.ndim == 3 else pred[b, :n])
    return out


def load_probe(path: Path, device) -> torch.nn.Module:
    ck = torch.load(path, map_location="cpu")
    cls, _ = PROBES[ck["probe"]]
    m = cls(C.BERT_DIM, ck["rank"])
    m.load_state_dict(ck["state_dict"])
    return m.to(device).eval()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--probe-dir", default=str(C.TASK2_OUT / "probes"))
    ap.add_argument("--out-dir", default=str(C.TASK2_OUT),
                    help="where the result files go (use a sub-folder for ablations, e.g. the spec-literal run)")
    args = ap.parse_args()
    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    C.set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pdir = Path(args.probe_dir)
    golds = {sp: GoldData(sp, device) for sp in ("dev", "test")}
    evals = {sp: SplitEval(g) for sp, g in golds.items()}

    rows, rel_rows, preds_cache = [], [], {}
    # LINEAR baseline (paper): distance |i - j|, depth = position
    for sp, ev in evals.items():
        dm = ev.distance_metrics([linear_chain_dist(len(s)) for s in ev.sents])
        dp = ev.depth_metrics([np.arange(len(s), dtype=float) for s in ev.sents])
        rows.append(dict(layer=-1, rank=0, probe="distance", split=sp, uuas=dm["uuas"], dspr=dm["dspr"], model="LINEAR"))
        rows.append(dict(layer=-1, rank=0, probe="depth", split=sp, nspr=dp["nspr"], root=dp["root"], model="LINEAR"))

    ckpts = sorted(pdir.glob("L*_*_k*.pt"))
    layers = sorted({int(p.name.split("_")[0][1:]) for p in ckpts})
    print(f"[eval] {len(ckpts)} probes over layers {layers}", flush=True)
    if not any(l >= 1 for l in layers):
        sys.exit(f"[eval] no trained probes for layers 1-12 in {pdir}. Probes are written when a layer "
                 f"finishes all its epochs: wait for '[train] all done' in the training log(s), then rerun.")
    for layer in layers:
        feats = {sp: LayerFeatures(g, layer) for sp, g in golds.items()}
        for path in sorted(pdir.glob(f"L{layer}_*_k*.pt")):
            ck = torch.load(path, map_location="cpu")
            model = load_probe(path, device)
            for sp, ev in evals.items():
                P = predict(model, feats[sp], golds[sp])
                base = dict(layer=layer, rank=ck["rank"], probe=ck["probe"], split=sp, model="BERT",
                            best_epoch=ck["best_epoch"], best_dev_loss=ck["best_dev_loss"])
                if ck["probe"] == "distance":
                    m = ev.distance_metrics(P, with_relations=(sp == "test"))
                    rows.append(dict(base, uuas=m["uuas"], dspr=m["dspr"]))
                    if sp == "test":
                        rel_rows += [dict(layer=layer, rank=ck["rank"], relation=k, **v) for k, v in m["relations"].items()]
                        preds_cache[(layer, "distance", ck["rank"])] = (P, m["per_sentence"])
                else:
                    m = ev.depth_metrics(P)
                    rows.append(dict(base, nspr=m["nspr"], root=m["root"]))
                    if sp == "test":
                        preds_cache[(layer, "depth", ck["rank"])] = (P, None)
            print(f"[eval] {path.name}: " + " ".join(
                f"{r['split']}:" + (f"UUAS={r['uuas']:.3f} DSpr={r['dspr']:.3f}" if r["probe"] == "distance"
                                    else f"NSpr={r['nspr']:.3f} Root={r['root']:.3f}")
                for r in rows[-2:]), flush=True)
        del feats
        torch.cuda.empty_cache() if device.type == "cuda" else None

    df = pd.DataFrame(rows)
    df.to_csv(out / "results_task2.csv", index=False)
    pd.DataFrame(rel_rows).to_csv(out / "per_relation_test.csv", index=False)

    # ---------------------------------------------------------------- l* selection on DEV
    bert = df[df.model == "BERT"]
    dist_dev = bert[(bert.probe == "distance") & (bert.split == "dev")]
    dep_dev = bert[(bert.probe == "depth") & (bert.split == "dev")]
    best_rank_d = dist_dev.loc[dist_dev.groupby("layer").uuas.idxmax()].set_index("layer")
    best_rank_n = dep_dev.loc[dep_dev.groupby("layer").nspr.idxmax()].set_index("layer")
    cand = best_rank_d.loc[[l for l in best_rank_d.index if l >= 1]]
    cand = cand.assign(nspr=best_rank_n.loc[cand.index, "nspr"])
    l_star = int(cand.sort_values(["uuas", "nspr"], ascending=False).index[0])
    k_d, k_n = int(best_rank_d.loc[l_star, "rank"]), int(best_rank_n.loc[l_star, "rank"])

    def get(layer, probe, rank, split):
        r = bert[(bert.layer == layer) & (bert.probe == probe) & (bert["rank"] == rank) & (bert.split == split)]
        return r.iloc[0].to_dict()

    test_d = bert[(bert.probe == "distance") & (bert.split == "test") & (bert.layer >= 1)]
    test_n = bert[(bert.probe == "depth") & (bert.split == "test") & (bert.layer >= 1)]
    best = dict(
        l_star=l_star, distance_rank=k_d, depth_rank=k_n,
        selection="dev UUAS (best rank per layer, layers 1-12), tie-break dev NSpr",
        dev=dict(uuas=get(l_star, "distance", k_d, "dev")["uuas"], dspr=get(l_star, "distance", k_d, "dev")["dspr"],
                 nspr=get(l_star, "depth", k_n, "dev")["nspr"], root=get(l_star, "depth", k_n, "dev")["root"]),
        test=dict(uuas=get(l_star, "distance", k_d, "test")["uuas"], dspr=get(l_star, "distance", k_d, "test")["dspr"],
                  nspr=get(l_star, "depth", k_n, "test")["nspr"], root=get(l_star, "depth", k_n, "test")["root"]),
        test_argmax_uuas_layer=int(test_d.loc[test_d.uuas.idxmax(), "layer"]),
        test_argmax_nspr_layer=int(test_n.loc[test_n.nspr.idxmax(), "layer"]),
        dev_argmax_nspr_layer=int(best_rank_n.loc[[l for l in best_rank_n.index if l >= 1], "nspr"].idxmax()),
    )
    (out / "best_layer.json").write_text(json.dumps(best, indent=2))

    keep_preds = {k: v for k, v in preds_cache.items()
                  if (k[0] == l_star and k[2] == (k_d if k[1] == "distance" else k_n)) or k[0] == 0}
    with open(out / "predictions_best.pkl", "wb") as f:
        pickle.dump(dict(l_star=l_star, k_distance=k_d, k_depth=k_n, preds=keep_preds,
                         sent_ids=[s.sent_id for s in evals["test"].sents]), f)

    # ---------------------------------------------------------------- paper comparison
    lin = df[(df.model == "LINEAR") & (df.split == "test")]
    comp = [dict(system=f"Ours: BERT-base-uncased L{l_star} (dev-selected), EWT test", uuas=100 * best["test"]["uuas"],
                 dspr=best["test"]["dspr"], root=100 * best["test"]["root"], nspr=best["test"]["nspr"]),
            dict(system="Ours: LINEAR baseline, EWT test", uuas=100 * lin[lin.probe == "distance"].uuas.iloc[0],
                 dspr=lin[lin.probe == "distance"].dspr.iloc[0], root=100 * lin[lin.probe == "depth"].root.iloc[0],
                 nspr=lin[lin.probe == "depth"].nspr.iloc[0])]
    comp += [dict(system=f"Paper: {k} (PTB WSJ test)", **v) for k, v in PAPER.items()]
    pd.DataFrame(comp).to_csv(out / "paper_comparison.csv", index=False)

    # ---------------------------------------------------------------- printout
    print("\n================ TASK 2 SUMMARY ================")
    print("Per layer (best rank chosen on dev; test metrics shown):")
    print(f"{'layer':>5} | {'k_d':>4} {'UUAS dev':>8} {'UUAS test':>9} {'DSpr test':>9} | {'k_n':>4} {'NSpr dev':>8} {'NSpr test':>9} {'Root test':>9}")
    for layer in sorted(best_rank_d.index):
        kd, kn = int(best_rank_d.loc[layer, "rank"]), int(best_rank_n.loc[layer, "rank"])
        td, tn = get(layer, "distance", kd, "test"), get(layer, "depth", kn, "test")
        print(f"{layer:>5} | {kd:>4} {best_rank_d.loc[layer, 'uuas']:8.3f} {td['uuas']:9.3f} {td['dspr']:9.3f} | "
              f"{kn:>4} {best_rank_n.loc[layer, 'nspr']:8.3f} {tn['nspr']:9.3f} {tn['root']:9.3f}")
    print("\nRank sweep at l* (test):")
    for probe, metric in (("distance", "uuas"), ("depth", "nspr")):
        sub = bert[(bert.layer == l_star) & (bert.probe == probe) & (bert.split == "test")].sort_values("rank")
        print(f"   {probe:8s} " + "  ".join(f"k={int(r['rank'])}: {r[metric]:.3f}" for _, r in sub.iterrows()))
    print(f"\nl* = {l_star} (distance k={k_d}, depth k={k_n}); test argmax UUAS layer = {best['test_argmax_uuas_layer']}, "
          f"test argmax NSpr layer = {best['test_argmax_nspr_layer']}, dev argmax NSpr layer = {best['dev_argmax_nspr_layer']}")
    print("\nPaper comparison (UUAS/Root% in %):")
    print(pd.DataFrame(comp).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print("================================================")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task2/extract_bert.py
"""Task 2, preparation: run bert-base-uncased ONCE over EWT train/dev/test and cache every
layer's word-level vectors (decision D8). Used by the probes (Task 2) and by Task 3.

Usage (from the project root; pick a GPU with >= 3 GB free):
    CUDA_VISIBLE_DEVICES=0 python code/task2/extract_bert.py            # extract + checks
    python code/task2/extract_bert.py --check-only                     # re-run checks only

Method:
  * Input = the gold EWT tokens (no re-tokenisation), passed with is_split_into_words=True.
    The uncased tokenizer lowercases internally. All sentences fit in 512 wordpieces (Phase 0).
  * output_hidden_states=True gives 13 tensors: [0] embeddings, [1..12] transformer layers.
  * word vector = mean of its wordpiece vectors (Hewitt & Manning 2019, footnote on BERT).
  * Computation in fp32 on GPU, stored as float16 (~5 GB in total).

Checks (written to outputs/task2/feature_checks.json):
  * alignment: #word vectors == #EWT tokens in every split; no NaN/Inf
  * contextuality per layer (Ethayarajh 2019): self-similarity of frequent words across their
    occurrences vs. the anisotropy baseline (mean cosine of random word-occurrence pairs)
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.data import load_ewt  # noqa: E402
from common.features import layer_path, load_layer, load_offsets  # noqa: E402

N_LAYERS = C.BERT_NUM_LAYERS + 1  # 13 incl. embeddings
CHECK_WORDS = ["the", "and", "to", "i", "it", "time", "people", "like", "right", "good", "work", "go"]


@torch.inference_mode()
def extract_split(split: str, tok, model, device, batch_size: int, out_dir: Path) -> dict:
    sents = load_ewt(split)
    lengths = np.array([len(s) for s in sents])
    offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
    n_words = int(offsets[-1])
    np.save(out_dir / f"{split}_offsets.npy", offsets)
    mm = [np.lib.format.open_memmap(layer_path(split, L, out_dir), mode="w+", dtype=np.float16,
                                    shape=(n_words, C.BERT_DIM)) for L in range(N_LAYERS)]

    order = np.argsort(-lengths, kind="stable")  # long first: similar lengths per batch, OOM early
    for start in tqdm(range(0, len(order), batch_size), desc=f"BERT {split}"):
        idx = order[start:start + batch_size]
        batch_tokens = [sents[i].tokens for i in idx]
        enc = tok(batch_tokens, is_split_into_words=True, return_tensors="pt", padding=True,
                  truncation=False)
        T = enc["input_ids"].shape[1]
        if T > C.BERT_MAX_WORDPIECES + 2:
            raise ValueError(f"sentence longer than 512 wordpieces in {split}")
        # word index of every wordpiece (-1 for [CLS]/[SEP]/[PAD])
        widx = torch.full((len(idx), T), -1, dtype=torch.long)
        for b in range(len(idx)):
            for t, w in enumerate(enc.word_ids(batch_index=b)):
                if w is not None:
                    widx[b, t] = w
        out = model(**{k: v.to(device) for k, v in enc.items()}, output_hidden_states=True)
        H = torch.stack(out.hidden_states, 0)  # [13, B, T, 768]

        widx = widx.to(device)
        max_w = int(lengths[idx].max())
        valid = widx >= 0
        tgt = torch.where(valid, widx, torch.full_like(widx, max_w))  # dump pieces into an extra slot
        sums = torch.zeros(N_LAYERS, len(idx), max_w + 1, C.BERT_DIM, device=device)
        sums.scatter_add_(2, tgt[None, :, :, None].expand(N_LAYERS, -1, -1, C.BERT_DIM), H.float())
        cnt = torch.zeros(len(idx), max_w + 1, device=device)
        cnt.scatter_add_(1, tgt, valid.float())
        if (cnt[:, :max_w] == 0).logical_and(
                torch.arange(max_w, device=device)[None] < torch.as_tensor(lengths[idx], device=device)[:, None]).any():
            raise RuntimeError("a word received zero wordpieces - alignment problem")
        mean = (sums[:, :, :max_w] / cnt[None, :, :max_w, None].clamp(min=1)).half().cpu().numpy()

        for b, s in enumerate(idx):
            n = lengths[s]
            for L in range(N_LAYERS):
                mm[L][offsets[s]:offsets[s] + n] = mean[L, b, :n]
    for m in mm:
        m.flush()
    meta = dict(split=split, n_sentences=len(sents), n_words=n_words, sent_ids=[s.sent_id for s in sents],
                model=C.BERT_MODEL, revision=C.BERT_REVISION, layers="0=embeddings, 1..12=transformer",
                pooling="mean over wordpieces", dtype="float16")
    (out_dir / f"{split}_meta.json").write_text(json.dumps(meta))
    return dict(n_sentences=len(sents), n_words=n_words)


def run_checks(out_dir: Path) -> dict:
    rng = np.random.default_rng(C.SEED)
    checks: dict = {"alignment": {}, "contextuality": {}}
    for split in C.SPLITS:
        offs = load_offsets(split, out_dir)
        n_tok = sum(len(s) for s in load_ewt(split))
        bad = 0
        for L in range(N_LAYERS):
            X = load_layer(split, L, feat_dir=out_dir)
            assert X.shape == (n_tok, C.BERT_DIM), (split, L, X.shape, n_tok)
            sample = np.asarray(X[rng.integers(0, len(X), 5000)], dtype=np.float32)
            bad += int((~np.isfinite(sample)).sum())
        checks["alignment"][split] = dict(n_words=int(offs[-1]), n_tokens=n_tok, match=int(offs[-1]) == n_tok,
                                          nonfinite_in_sample=bad)

    # contextuality on train: self-similarity vs anisotropy (Ethayarajh 2019)
    words = [w for s in load_ewt("train") for w in s.lower]
    pos: dict[str, np.ndarray] = {}
    arr = np.array(words, dtype=object)
    for w in CHECK_WORDS:
        p = np.flatnonzero(arr == w)
        if len(p) >= 20:
            pos[w] = rng.choice(p, size=min(200, len(p)), replace=False)
    pairs = rng.integers(0, len(words), size=(5000, 2))
    print("\nlayer | anisotropy (random pairs) | mean self-sim | self-sim - anisotropy")
    for L in range(N_LAYERS):
        X = load_layer("train", L, feat_dir=out_dir)

        def unit(v):
            v = np.asarray(v, dtype=np.float32)
            return v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-12)

        a, b = unit(X[np.sort(pairs[:, 0])]), unit(X[np.sort(pairs[:, 1])])
        aniso = float(np.mean(np.sum(a * b, 1)))
        selfsim = {}
        for w, p in pos.items():
            U = unit(X[np.sort(p)])
            G = U @ U.T
            k = len(U)
            selfsim[w] = float((G.sum() - k) / (k * (k - 1)))
        ms = float(np.mean(list(selfsim.values())))
        checks["contextuality"][L] = dict(anisotropy=aniso, self_sim=selfsim, mean_self_sim=ms,
                                          adjusted_self_sim=ms - aniso)
        print(f"{L:5d} | {aniso:25.3f} | {ms:13.3f} | {ms - aniso:8.3f}")
    return checks


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--check-only", action="store_true")
    ap.add_argument("--splits", nargs="+", default=C.SPLITS)
    args = ap.parse_args()
    C.set_seed()
    out_dir = C.FEATURE_DIR
    out_dir.mkdir(parents=True, exist_ok=True)

    if not args.check_only:
        from transformers import AutoModel, AutoTokenizer

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[extract] device={device} model={C.BERT_MODEL}@{C.BERT_REVISION[:7]} -> {out_dir}")
        tok = AutoTokenizer.from_pretrained(C.BERT_MODEL, revision=C.BERT_REVISION)
        model = AutoModel.from_pretrained(C.BERT_MODEL, revision=C.BERT_REVISION).to(device).eval()
        t0 = time.time()
        for split in args.splits:
            info = extract_split(split, tok, model, device, args.batch_size, out_dir)
            print(f"[extract] {split}: {info}")
        print(f"[extract] done in {time.time() - t0:.0f} s")

    checks = run_checks(out_dir)
    print("\n[alignment]", json.dumps(checks["alignment"], indent=1))
    (C.TASK2_OUT / "feature_checks.json").write_text(json.dumps(checks, indent=1))
    size = sum(f.stat().st_size for f in out_dir.glob("*.npy")) / 1e9
    print(f"[extract] feature dir size: {size:.2f} GB; checks -> {C.TASK2_OUT / 'feature_checks.json'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task2/layer_significance.py
"""Task 2 add-on: is the dev-selected l* significantly better than its neighbouring layers?

The test UUAS curve is a plateau over layers 6-8 (79.6 / 79.5 / 79.5). This script runs a
paired bootstrap over sentences (same sentences for both layers) for every layer vs l*, on dev
and on test, so the report can state whether "l* = 8" is a real peak or one point of a plateau.

Usage (from the project root, after train_probes.py / evaluate_task2.py):
    python code/task2/layer_significance.py
Output: outputs/task2/layer_significance.csv
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "task3"))
from common import config as C  # noqa: E402
from common.data import load_ewt  # noqa: E402
from common.features import load_layer, load_meta, load_offsets  # noqa: E402
from common.stats import bootstrap_uuas, paired_bootstrap_uuas  # noqa: E402
from common.trees import gold_mst_edges, mst_on_subset  # noqa: E402

from t3_common import probe_matrix, probe_sq_dist  # noqa: E402


def per_sentence_counts(split: str, layer: int, rank: int, sents, gold) -> tuple[np.ndarray, np.ndarray]:
    X = load_layer(split, layer, mmap=True)
    off = load_offsets(split)
    Bm = probe_matrix(layer, rank)
    corr, tot = np.zeros(len(sents)), np.zeros(len(sents))
    for i, s in enumerate(sents):
        keep = s.nonpunct_mask
        if keep.sum() < 2:
            continue
        h = np.asarray(X[off[i]:off[i + 1]], dtype=np.float64)
        pred = mst_on_subset(probe_sq_dist(h, Bm), keep)
        corr[i], tot[i] = len(pred & gold[i]), len(gold[i])
    return corr, tot


def main() -> None:
    bl = json.loads((C.TASK2_OUT / "best_layer.json").read_text())
    L, k = bl["l_star"], bl["distance_rank"]
    res = pd.read_csv(C.TASK2_OUT / "results_task2.csv")
    rows = []
    for split in ("dev", "test"):
        sents = load_ewt(split)
        assert load_meta(split)["sent_ids"] == [s.sent_id for s in sents]
        gold = [gold_mst_edges(s.heads, s.nonpunct_mask) if s.nonpunct_mask.sum() >= 2 else set() for s in sents]
        ref_c, tot = per_sentence_counts(split, L, k, sents, gold)
        # consistency check against evaluate_task2.py
        expected = res[(res.model == "BERT") & (res.probe == "distance") & (res.layer == L) &
                       (res["rank"] == k) & (res.split == split)].uuas.iloc[0]
        # (float64 here vs float32 on GPU there: allow a rare MST tie flip)
        assert abs(ref_c.sum() / tot.sum() - expected) < 5e-4, (ref_c.sum() / tot.sum(), expected)
        print(f"[{split}] check vs evaluate_task2: {ref_c.sum() / tot.sum():.5f} vs {expected:.5f}", flush=True)
        for layer in range(C.BERT_NUM_LAYERS + 1):
            c = ref_c if layer == L else per_sentence_counts(split, layer, k, sents, gold)[0]
            t = paired_bootstrap_uuas(c, ref_c, tot) if layer != L else dict(diff=0.0, ci_low=0.0, ci_high=0.0,
                                                                             frac_le_zero=float("nan"))
            lo, hi = bootstrap_uuas(c, tot)
            rows.append(dict(split=split, layer=layer, rank=k, uuas=c.sum() / tot.sum(), ci_low=lo, ci_high=hi,
                             diff_vs_lstar=t["diff"], diff_ci_low=t["ci_low"], diff_ci_high=t["ci_high"],
                             significant=bool(t["ci_high"] < 0 or t["ci_low"] > 0)))
            r = rows[-1]
            print(f"[{split}] L{layer:2d} UUAS={r['uuas']:.4f} [{lo:.4f},{hi:.4f}]  vs L{L}: {r['diff_vs_lstar']:+.4f} "
                  f"[{r['diff_ci_low']:+.4f},{r['diff_ci_high']:+.4f}]{' *' if r['significant'] else ''}", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(C.TASK2_OUT / "layer_significance.csv", index=False)
    for split in ("dev", "test"):
        d = df[df.split == split]
        tied = d[~d.significant].layer.tolist()
        print(f"[{split}] layers statistically indistinguishable from l*={L} (95% paired CI covers 0): {tied}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task2/plot_task2.py
"""Task 2 figures (PNG for slides + PDF for the LaTeX report) and qualitative examples.

Usage (from the project root, after evaluate_task2.py):
    python code/task2/plot_task2.py

Figures -> outputs/task2/figures/:
  uuas_vs_layer          test UUAS vs layer, one line per probe rank, LINEAR baseline, l* marked
  nspr_vs_layer          depth-probe Spearman (NSpr) vs layer, same layout
  metrics_vs_layer       2x2 small multiples: UUAS, DSpr, NSpr, Root%
  relation_recall_heatmap  distance-probe recall per dependency relation x layer (best rank)
  example_<kind>_*       gold vs predicted tree arcs for a good / typical / poor test sentence
  distance_heatmap_*     gold tree distances vs predicted squared distances (paper Fig. 4 style)
  depth_example_*        gold vs predicted depth per word
qualitative_examples.json : the chosen sentences with missed / spurious edges (for the report)
"""
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.patches import Arc  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.data import gold_depths, gold_distance_matrix, load_ewt  # noqa: E402
from common.trees import gold_mst_edges, mst_on_subset  # noqa: E402

# ---- palette (dataviz reference instance, light surface; fixed categorical order) ----
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # slots 1-3: blue, orange, aqua
MARKERS = ["o", "s", "^"]  # secondary encoding: identity never colour-only
SEQ_CMAP = "Blues"  # single-hue sequential for magnitudes
OUT = C.TASK2_OUT / "figures"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False, "lines.linewidth": 1.6,
})


def save(fig, name: str) -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


def metric_panel(ax, df, probe, metric, l_star, lin_value, ylabel, scale=1.0, label_lines=True):
    sub = df[(df.model == "BERT") & (df.probe == probe) & (df.split == "test")]
    ranks = sorted(sub["rank"].unique())
    ends = []
    for i, k in enumerate(ranks):
        s = sub[sub["rank"] == k].sort_values("layer")
        ax.plot(s.layer, s[metric] * scale, color=SERIES[i], marker=MARKERS[i], markersize=5,
                label=f"rank k={k}")
        ends.append((s[metric].iloc[-1] * scale, f"k={k}", s.layer.iloc[-1]))
    if lin_value is not None:
        ax.axhline(lin_value * scale, color=INK2, linestyle="--", linewidth=1.0)
        ax.annotate("LINEAR baseline", (0, lin_value * scale), xytext=(2, 4), textcoords="offset points",
                    color=INK2, fontsize=8)
    ax.axvline(l_star, color=INK2, linestyle=":", linewidth=1.0)
    lo, hi = ax.get_ylim()
    hi = hi + 0.10 * (hi - lo)  # headroom above the data so the ℓ* label never sits on a marker
    ax.set_ylim(lo, hi)
    ax.annotate(f"ℓ* = {l_star} (dev)", (l_star, hi), xytext=(4, -10), textcoords="offset points",
                color=INK2, fontsize=8)
    if label_lines:  # direct labels at the right end, dodged so they never overlap
        gap = 0.045 * (hi - lo)
        placed = []
        for y, txt, x in sorted(ends):
            y_lab = max(y, placed[-1] + gap) if placed else y
            placed.append(y_lab)
            ax.annotate(txt, (x, y), xytext=(x + 0.25, y_lab), textcoords="data", color=INK2, fontsize=8,
                        va="center", annotation_clip=False)
    ax.set_xticks(range(0, 13))
    ax.set_xlabel("BERT layer ℓ (0 = embeddings)")
    ax.set_ylabel(ylabel)


def arc_diagram(ax, tokens, gold, pred, title, pred_label: str = "probe MST"):
    """Gold edges as arcs above the words, predicted MST edges below.
    Predicted edges: blue = correct, orange dashed = wrong (not in gold)."""
    n = len(tokens)
    for i, t in enumerate(tokens):
        ax.text(i, 0, t, ha="center", va="center", fontsize=8, rotation=0, color=INK)
    for a, b in gold:
        ax.add_patch(Arc(((a + b) / 2, 0.35), b - a, 0.6 * (b - a) ** 0.8 + 0.4, theta1=0, theta2=180,
                         color=INK2, linewidth=1.2))
    for a, b in pred:
        ok = (a, b) in gold
        ax.add_patch(Arc(((a + b) / 2, -0.35), b - a, 0.6 * (b - a) ** 0.8 + 0.4, theta1=180, theta2=360,
                         color=SERIES[0] if ok else SERIES[1], linewidth=1.6 if ok else 1.4,
                         linestyle="-" if ok else "--"))
    h = 0.6 * max(1, n) ** 0.8 / 2 + 1
    ax.set_xlim(-1, n)
    ax.set_ylim(-h, h)
    ax.axis("off")
    ax.text(-1, h * 0.9, "gold tree (above)", fontsize=8, color=INK2)
    ax.text(-1, -h * 0.95, f"{pred_label} (below): solid blue = correct, dashed orange = wrong", fontsize=8, color=INK2)
    ax.set_title(title, fontsize=10, loc="left")


def main() -> None:
    df = pd.read_csv(C.TASK2_OUT / "results_task2.csv")
    best = json.loads((C.TASK2_OUT / "best_layer.json").read_text())
    l_star = best["l_star"]
    lin = df[(df.model == "LINEAR") & (df.split == "test")]
    lin_d, lin_n = lin[lin.probe == "distance"].iloc[0], lin[lin.probe == "depth"].iloc[0]

    # 1-2. headline single-metric figures (slides)
    for probe, metric, lv, ylabel, scale, name in (
            ("distance", "uuas", lin_d.uuas, "UUAS on EWT test (%)", 100, "uuas_vs_layer"),
            ("depth", "nspr", lin_n.nspr, "Depth Spearman (NSpr), EWT test", 1, "nspr_vs_layer")):
        fig, ax = plt.subplots(figsize=(6.4, 3.8))
        metric_panel(ax, df, probe, metric, l_star, lv, ylabel, scale)
        ax.set_title(f"{'Distance' if probe == 'distance' else 'Depth'} probe on bert-base-uncased: {ylabel.split(' (')[0].split(',')[0]} by layer",
                     loc="left")
        ax.legend(loc="best", fontsize=8)
        save(fig, name)

    # 3. 2x2 small multiples (report)
    fig, axes = plt.subplots(2, 2, figsize=(10, 6.6), sharex=True)
    for ax, (probe, metric, lv, ylabel, scale) in zip(axes.ravel(), (
            ("distance", "uuas", lin_d.uuas, "UUAS (%)", 100), ("distance", "dspr", lin_d.dspr, "DSpr.", 1),
            ("depth", "nspr", lin_n.nspr, "NSpr. (depth Spearman)", 1), ("depth", "root", lin_n.root, "Root (%)", 100))):
        metric_panel(ax, df, probe, metric, l_star, lv, ylabel, scale, label_lines=False)
        ax.set_title(ylabel, loc="left")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.suptitle("Structural probes on EWT test, by BERT layer (dashed = LINEAR baseline, dotted = dev-selected ℓ*)",
                 x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.legend(handles, labels, loc="upper right", ncol=3, fontsize=8, bbox_to_anchor=(1.0, 0.965))
    save(fig, "metrics_vs_layer")

    # 4. per-relation recall heatmap (best dev rank per layer)
    rel = pd.read_csv(C.TASK2_OUT / "per_relation_test.csv")
    dev_d = df[(df.model == "BERT") & (df.probe == "distance") & (df.split == "dev")]
    best_rank = dev_d.loc[dev_d.groupby("layer").uuas.idxmax()].set_index("layer")["rank"]
    rel = rel[rel.apply(lambda r: best_rank.get(r.layer) == r["rank"], axis=1)]
    top = rel[rel.layer == l_star].sort_values("total", ascending=False).relation.head(14).tolist()
    M = rel[rel.relation.isin(top)].pivot(index="relation", columns="layer", values="recall").loc[top]
    fig, ax = plt.subplots(figsize=(8.5, 5))
    im = ax.imshow(M.values, cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(M.shape[1]), M.columns)
    ax.set_yticks(range(M.shape[0]), [f"{r} (n={int(rel[(rel.layer == l_star) & (rel.relation == r)].total.iloc[0])})" for r in top])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            ax.text(j, i, f"{v:.2f}"[1:] if v < 1 else "1.0", ha="center", va="center", fontsize=7,
                    color="white" if v > 0.6 else INK)
    ax.grid(False)
    ax.set_xlabel("BERT layer ℓ")
    ax.set_title("Distance-probe recall of gold edges, by dependency relation (EWT test)", loc="left")
    fig.colorbar(im, ax=ax, fraction=0.03, label="recall")
    save(fig, "relation_recall_heatmap")

    # 5-7. qualitative examples at l*
    with open(C.TASK2_OUT / "predictions_best.pkl", "rb") as f:
        P = pickle.load(f)
    test = load_ewt("test")
    dist_preds, per_sent = P["preds"][(l_star, "distance", P["k_distance"])]
    depth_preds, _ = P["preds"][(l_star, "depth", P["k_depth"])]
    cand = [i for i, s in enumerate(test) if 10 <= s.nonpunct_mask.sum() <= 18]
    scores = {i: per_sent[i][0] / max(per_sent[i][1], 1) for i in cand}
    order = sorted(cand, key=lambda i: scores[i])
    picks = {"good": order[-1], "typical": order[len(order) // 2], "poor": order[0]}
    qual = {}
    for kind, i in picks.items():
        s = test[i]
        keep = s.nonpunct_mask
        idx = np.flatnonzero(keep)
        remap = {int(o): j for j, o in enumerate(idx)}
        gold = {tuple(sorted((remap[a], remap[b]))) for a, b in gold_mst_edges(s.heads, keep)}
        pred = {tuple(sorted((remap[a], remap[b]))) for a, b in mst_on_subset(dist_preds[i], keep)}
        toks = [s.tokens[o] for o in idx]
        fig, ax = plt.subplots(figsize=(max(7, 0.75 * len(toks)), 3.6))
        arc_diagram(ax, toks, gold, pred, f"{kind.capitalize()} example, layer {l_star}: UUAS {scores[i]:.2f}  ({s.sent_id})")
        save(fig, f"example_{kind}")
        rel_of = {tuple(sorted((remap[d], remap[h - 1]))): s.deprels[d].split(":")[0]
                  for d, h in enumerate(s.heads) if h > 0 and keep[d] and keep[h - 1]}
        qual[kind] = dict(sent_id=s.sent_id, text=s.text, uuas=scores[i],
                          missed=[f"{toks[a]}-{toks[b]} ({rel_of.get((a, b), '?')})" for a, b in sorted(gold - pred)],
                          spurious=[f"{toks[a]}-{toks[b]}" for a, b in sorted(pred - gold)])

    i = picks["typical"]
    s = test[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
    for ax, M, t in ((axes[0], gold_distance_matrix(s.heads), "gold tree distance"),
                     (axes[1], dist_preds[i], f"predicted squared distance (layer {l_star})")):
        im = ax.imshow(M, cmap=SEQ_CMAP + "_r")
        ax.set_xticks(range(len(s)), s.tokens, rotation=90, fontsize=7)
        ax.set_yticks(range(len(s)), s.tokens, fontsize=7)
        ax.grid(False)
        ax.set_title(t, loc="left")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Darker = closer in the tree. The probe recovers the block structure of the gold tree.", x=0.01, ha="left")
    fig.tight_layout()
    save(fig, "distance_heatmap_typical")

    fig, ax = plt.subplots(figsize=(max(6.5, 0.5 * len(s)), 3.4))
    x = np.arange(len(s))
    ax.plot(x, gold_depths(s.heads), color=INK2, marker="o", markersize=5, label="gold depth")
    ax.plot(x, depth_preds[i], color=SERIES[0], marker="s", markersize=5, label=f"predicted depth (layer {l_star})")
    ax.set_xticks(x, s.tokens, rotation=60, fontsize=8)
    ax.set_ylabel("depth (edges to root)")
    ax.set_title("Depth probe: gold vs predicted depth per word (typical test sentence)", loc="left")
    ax.legend(fontsize=8)
    save(fig, "depth_example_typical")

    (C.TASK2_OUT / "qualitative_examples.json").write_text(json.dumps(qual, indent=2))
    print(f"[plot] figures -> {OUT}")
    for k, v in qual.items():
        print(f"\n[{k}] UUAS={v['uuas']:.2f}  {v['text']}\n   missed:   {v['missed']}\n   spurious: {v['spurious']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task2/probe_data.py
"""Batching of cached BERT word features + gold tree targets for the structural probes.

GoldData (per split, layer-independent): gold distance matrices, gold depths, lengths, offsets.
LayerFeatures (per split and layer): the [n_words, 768] matrix moved to the GPU once
(~0.3 GB for train in fp16), plus one zero row used as padding.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.data import gold_depths, gold_distance_matrix, load_ewt  # noqa: E402
from common.features import load_layer, load_offsets  # noqa: E402


class GoldData:
    def __init__(self, split: str, device: torch.device):
        self.split = split
        self.sents = load_ewt(split)
        self.offsets = load_offsets(split)
        self.lengths = np.array([len(s) for s in self.sents])
        assert self.offsets[-1] == self.lengths.sum(), "features and EWT split are misaligned"
        self.device = device
        self.dist = [torch.from_numpy(gold_distance_matrix(s.heads)).to(device) for s in self.sents]
        self.depth = [torch.from_numpy(gold_depths(s.heads)).to(device) for s in self.sents]
        self.offsets_t = torch.as_tensor(self.offsets[:-1], device=device)
        self.lengths_t = torch.as_tensor(self.lengths, device=device)

    def __len__(self) -> int:
        return len(self.sents)

    def targets(self, idx: torch.Tensor):
        """Padded gold distances [B, L, L] and depths [B, L] (padding = -1), lengths [B]."""
        lens = self.lengths_t[idx]
        L = int(lens.max())
        B = len(idx)
        dist = torch.full((B, L, L), -1.0, device=self.device)
        dep = torch.full((B, L), -1.0, device=self.device)
        for b, s in enumerate(idx.tolist()):
            n = self.lengths[s]
            dist[b, :n, :n] = self.dist[s]
            dep[b, :n] = self.depth[s]
        return dist, dep, lens


class LayerFeatures:
    def __init__(self, gold: GoldData, layer: int):
        X = np.asarray(load_layer(gold.split, layer, mmap=True))
        X = np.concatenate([X, np.zeros((1, X.shape[1]), dtype=X.dtype)])  # padding row
        self.X = torch.from_numpy(X).to(gold.device)  # fp16 on GPU
        self.pad_row = X.shape[0] - 1
        self.gold = gold

    def inputs(self, idx: torch.Tensor) -> torch.Tensor:
        """[B, L, 768] float32 word vectors for the sentences in idx (zero-padded)."""
        lens = self.gold.lengths_t[idx]
        L = int(lens.max())
        ar = torch.arange(L, device=self.X.device)
        rows = self.gold.offsets_t[idx][:, None] + ar[None, :]
        rows = torch.where(ar[None, :] < lens[:, None], rows, torch.full_like(rows, self.pad_row))
        return self.X[rows].float()


In [ ]:
%%writefile code/task2/probes.py
"""Structural probes and losses (Hewitt & Manning 2019; mirrors structural-probes/probe.py + loss.py).

Distance probe (TwoWordPSDProbe):  d_B(h_i, h_j)^2 = || B (h_i - h_j) ||^2  ~  tree distance d(w_i, w_j)
Depth probe    (OneWordPSDProbe):  || B h_i ||^2                         ~  tree depth ||w_i||

B is stored as `proj` with shape [768, k] (repo convention: transformed = h @ proj), initialised
U(-0.05, 0.05). The SQUARED distance / norm is what is trained to match the gold value (paper
App. A.1).

Losses (paper App. A.2, repo loss.py): L1 between prediction and gold, padded positions masked.
  distance: per sentence  sum_{i,j} |pred - gold| / n^2
  depth:    per sentence  sum_i     |pred - gold| / n
  then averaged over the sentences in the batch.
"""
from __future__ import annotations

import torch
from torch import nn


class DistanceProbe(nn.Module):
    def __init__(self, dim: int, rank: int):
        super().__init__()
        self.proj = nn.Parameter(torch.empty(dim, rank).uniform_(-0.05, 0.05))

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """h: [B, L, dim] -> predicted squared distances [B, L, L]."""
        t = h @ self.proj  # [B, L, k]
        # ||t_i - t_j||^2 = ||t_i||^2 + ||t_j||^2 - 2 t_i.t_j : same value as the explicit
        # difference, but O(B L^2) memory instead of O(B L^2 k). Clamp removes tiny negative
        # round-off on the diagonal.
        sq = (t * t).sum(-1)  # [B, L]
        d = sq.unsqueeze(2) + sq.unsqueeze(1) - 2.0 * (t @ t.transpose(1, 2))
        return d.clamp_min(0.0)


class DepthProbe(nn.Module):
    def __init__(self, dim: int, rank: int):
        super().__init__()
        self.proj = nn.Parameter(torch.empty(dim, rank).uniform_(-0.05, 0.05))

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """h: [B, L, dim] -> predicted squared norms (depths) [B, L]."""
        t = h @ self.proj
        return (t * t).sum(-1)


def distance_loss(pred: torch.Tensor, gold: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    """pred/gold: [B, L, L]; gold = -1 on padding. Returns the batch-mean of per-sentence losses."""
    mask = (gold >= 0).float()
    per_sent = (torch.abs(pred - gold) * mask).sum((1, 2)) / lengths.float().pow(2)
    return per_sent.mean()


def depth_loss(pred: torch.Tensor, gold: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    """pred/gold: [B, L]; gold = -1 on padding."""
    mask = (gold >= 0).float()
    per_sent = (torch.abs(pred - gold) * mask).sum(1) / lengths.float()
    return per_sent.mean()


PROBES = {"distance": (DistanceProbe, distance_loss), "depth": (DepthProbe, depth_loss)}


In [ ]:
%%writefile code/task2/train_probes.py
"""Task 2: train structural probes for every BERT layer x rank x {distance, depth}.

Usage (from the project root):
    # smoke test (seconds)
    CUDA_VISIBLE_DEVICES=0 python code/task2/train_probes.py --layers 7 --ranks 64 --epochs 1 --tag smoke
    # full sweep: layers 0..12 x ranks {32,64,128} x {distance, depth} = 78 probes, 30 epochs each
    CUDA_VISIBLE_DEVICES=0 nohup python code/task2/train_probes.py > outputs/task2_train.log 2>&1 &

Training (assignment step 3 + paper App. A.2):
  * Adam, lr 1e-3, batch 20 sentences (random order, re-shuffled every epoch), 30 epochs, train split.
  * Paper schedule (decision D7): after each epoch the dev loss is computed. If it does not reach a
    new minimum (by > 1e-4), the learning rate is multiplied by 0.1 and the optimizer is reset (no
    momentum kept). The checkpoint with the lowest dev loss is kept.
  * All 6 probes of a layer are trained side by side on the same batches (independent parameters,
    optimizers and schedules), so each batch is built only once.
  * Resumable: probes whose finished checkpoint exists are skipped.

Outputs: outputs/task2/probes[/<tag>]/L{layer}_{probe}_k{rank}.pt  (best state + training history)
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402

from probe_data import GoldData, LayerFeatures  # noqa: E402
from probes import PROBES  # noqa: E402

MIN_DELTA = 1e-4


def ckpt_path(out: Path, layer: int, probe: str, rank: int) -> Path:
    return out / f"L{layer}_{probe}_k{rank}.pt"


class ProbeRun:
    """One probe with its own optimizer, LR schedule and best-dev checkpoint."""

    def __init__(self, layer: int, probe: str, rank: int, device, lr: float):
        torch.manual_seed(C.SEED + 1000 * layer + rank + (0 if probe == "distance" else 500))
        cls, self.loss_fn = PROBES[probe]
        self.model = cls(C.BERT_DIM, rank).to(device)
        self.layer, self.probe, self.rank, self.lr = layer, probe, rank, lr
        self.opt = torch.optim.Adam(self.model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
        self.best, self.best_epoch, self.best_state = float("inf"), -1, None
        self.history: list[dict] = []
        self._train_sum, self._train_n = 0.0, 0

    def step(self, h, dist, dep, lens) -> None:
        pred = self.model(h)
        loss = self.loss_fn(pred, dist if self.probe == "distance" else dep, lens)
        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        self.opt.step()
        self._train_sum = self._train_sum + loss.detach() * len(lens)  # stays on GPU: no per-step sync
        self._train_n += len(lens)

    @torch.no_grad()
    def dev_loss(self, dev_batches) -> float:
        tot, n = 0.0, 0
        for h, dist, dep, lens in dev_batches():
            pred = self.model(h)
            tot += self.loss_fn(pred, dist if self.probe == "distance" else dep, lens).item() * len(lens)
            n += len(lens)
        return tot / n

    def end_epoch(self, epoch: int, dev: float, decay: bool = True, keep_last: bool = False) -> None:
        improved = dev < self.best - MIN_DELTA
        self.history.append(dict(epoch=epoch, train_loss=float(self._train_sum) / max(self._train_n, 1),
                                 dev_loss=dev, lr=self.lr, improved=improved))
        self._train_sum, self._train_n = 0.0, 0
        if improved:
            self.best, self.best_epoch = dev, epoch
            if not keep_last:
                self.best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
        if keep_last:  # literal assignment: the saved probe is the final-epoch one (overwritten each epoch)
            self.best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
            self.last_epoch, self.last_dev = epoch, dev
        if not improved and decay:  # paper: reset optimizer with lr * 0.1
            self.lr *= 0.1
            self.opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, betas=(0.9, 0.999), eps=1e-8)

    def save(self, path: Path, args) -> None:
        # best_* = the saved state: the best-dev epoch, or (keep_last) the final epoch
        saved_epoch, saved_dev = ((self.last_epoch, self.last_dev) if args.keep_last
                                  else (self.best_epoch, self.best))
        torch.save(dict(state_dict=self.best_state, layer=self.layer, probe=self.probe, rank=self.rank,
                        best_dev_loss=saved_dev, best_epoch=saved_epoch, min_dev_loss=self.best,
                        min_dev_epoch=self.best_epoch, history=self.history,
                        epochs=args.epochs, batch_size=args.batch_size, lr0=args.lr, seed=C.SEED,
                        lr_decay=not args.no_decay, keep_last=args.keep_last, finished=True), path)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--layers", type=int, nargs="+", default=C.PROBE_LAYERS)
    ap.add_argument("--ranks", type=int, nargs="+", default=C.PROBE_RANKS)
    ap.add_argument("--probes", nargs="+", default=["distance", "depth"])
    ap.add_argument("--epochs", type=int, default=C.PROBE_EPOCHS)
    ap.add_argument("--batch-size", type=int, default=C.PROBE_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=C.PROBE_LR)
    ap.add_argument("--tag", default=None, help="write to outputs/task2/probes_<tag>/ (e.g. smoke)")
    ap.add_argument("--max-train-sents", type=int, default=None, help="debug: subsample train")
    ap.add_argument("--no-decay", action="store_true",
                    help="constant lr (no x0.1 decay on dev plateau): the literal assignment schedule")
    ap.add_argument("--keep-last", action="store_true",
                    help="save the final-epoch probe instead of the best-dev one (literal assignment)")
    args = ap.parse_args()
    C.set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = C.TASK2_OUT / (f"probes_{args.tag}" if args.tag else "probes")
    out.mkdir(parents=True, exist_ok=True)
    print(f"[train] device={device} layers={args.layers} ranks={args.ranks} probes={args.probes} "
          f"epochs={args.epochs} bs={args.batch_size} lr={args.lr} decay={not args.no_decay} "
          f"keep_last={args.keep_last} -> {out}", flush=True)

    t0 = time.time()
    train_gold, dev_gold = GoldData("train", device), GoldData("dev", device)
    print(f"[train] gold targets ready ({time.time() - t0:.0f} s)", flush=True)
    n_train = len(train_gold) if args.max_train_sents is None else min(args.max_train_sents, len(train_gold))

    for layer in args.layers:
        todo = [(p, k) for p in args.probes for k in args.ranks if not ckpt_path(out, layer, p, k).exists()]
        if not todo:
            print(f"[train] layer {layer}: all probes done, skipping", flush=True)
            continue
        tl = time.time()
        train_x, dev_x = LayerFeatures(train_gold, layer), LayerFeatures(dev_gold, layer)
        runs = [ProbeRun(layer, p, k, device, args.lr) for p, k in todo]

        dev_idx = torch.arange(len(dev_gold), device=device)

        def dev_batches():
            for s in range(0, len(dev_idx), 100):
                idx = dev_idx[s:s + 100]
                dist, dep, lens = dev_gold.targets(idx)
                yield dev_x.inputs(idx), dist, dep, lens

        gen = torch.Generator().manual_seed(C.SEED + layer)
        for epoch in range(1, args.epochs + 1):
            perm = torch.randperm(n_train, generator=gen).to(device)
            for r in runs:
                r.model.train()
            for s in range(0, n_train, args.batch_size):
                idx = perm[s:s + args.batch_size]
                dist, dep, lens = train_gold.targets(idx)
                h = train_x.inputs(idx)
                for r in runs:
                    r.step(h, dist, dep, lens)
            msg = []
            for r in runs:
                r.model.eval()
                r.end_epoch(epoch, r.dev_loss(dev_batches), decay=not args.no_decay, keep_last=args.keep_last)
                hst = r.history[-1]
                msg.append(f"{r.probe[:4]}-k{r.rank}: tr={hst['train_loss']:.4f} dev={hst['dev_loss']:.4f}"
                           f"{'*' if hst['improved'] else ''} lr={hst['lr']:.0e}")
            print(f"[L{layer:02d} ep{epoch:02d}] " + " | ".join(msg), flush=True)

        for r in runs:
            r.save(ckpt_path(out, layer, r.probe, r.rank), args)
        print(f"[train] layer {layer} done in {time.time() - tl:.0f} s; best dev: " +
              ", ".join(f"{r.probe}-k{r.rank}={r.best:.4f}@ep{r.best_epoch}" for r in runs), flush=True)
        del train_x, dev_x, runs
        if device.type == "cuda":
            torch.cuda.empty_cache()
    (out / "train_config.json").write_text(json.dumps(vars(args), indent=1))
    print(f"[train] all done in {(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


`layer_significance.py` (the last Task 2 step) reuses the probe helpers of Task 3, so that one module is written here.

In [ ]:
%%writefile code/task3/t3_common.py
"""Helpers shared by the Task 3 scripts.

* load_t1 / load_t3_static : the two static vector sets (same vocabulary and row order)
* abtt                     : "all-but-the-top" post-processing (Mu & Viswanath, ICLR 2018)
* anisotropy               : mean cosine between random word-type pairs (Ethayarajh 2019)
* ContextualTest           : raw in-context BERT word vectors for test sentences (Task 2 cache)
* probe_matrix / probe_sq_dist : the trained Task 2 distance probe B, applied to any vectors
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.data import Sentence  # noqa: E402
from common.features import load_layer, load_meta, load_offsets  # noqa: E402

STATIC_DIR = C.TASK3_OUT / "static"


def load_w2i() -> dict[str, int]:
    vocab = json.loads((C.TASK1_OUT / "vocab.json").read_text())
    return {w: i for i, w in enumerate(vocab)}


def load_t1(tag: str | None = None) -> np.ndarray:
    """T1 PPMI-SVD vectors: the spec run (tag=None) or an ablation (e.g. 'p0')."""
    d = C.TASK1_OUT if tag is None else C.TASK1_OUT / "ablations" / tag
    return np.load(d / "vectors.npy").astype(np.float32)


def t1_dir(tag: str | None = None) -> Path:
    return C.TASK1_OUT if tag is None else C.TASK1_OUT / "ablations" / tag


def load_t3_static(layer: int) -> np.ndarray:
    return np.load(STATIC_DIR / f"layer{layer}.npy").astype(np.float32)


def best_layer() -> dict:
    return json.loads((C.TASK2_OUT / "best_layer.json").read_text())


# --------------------------------------------------------------------------- post-processing
def abtt(V: np.ndarray, n_components: int | None = None) -> np.ndarray:
    """All-but-the-top: subtract the mean vector, then remove the projections on the top-D
    principal directions. D = d/100 by default (the paper's rule of thumb: 7 for BERT's 768
    dims, 1 for the 100-d SVD vectors). Zero rows (words with no context) stay zero."""
    V = np.asarray(V, dtype=np.float64)
    D = max(1, V.shape[1] // 100) if n_components is None else n_components
    nz = np.linalg.norm(V, axis=1) > 0
    mu = V[nz].mean(0)
    Xc = V - mu
    # top-D right singular vectors of the centred (non-zero) rows = top-D principal directions
    _, _, Vt = np.linalg.svd(Xc[nz], full_matrices=False)
    U = Vt[:D].T  # [d, D]
    out = Xc - (Xc @ U) @ U.T
    out[~nz] = 0.0
    return out.astype(np.float32)


def anisotropy(V: np.ndarray, n_pairs: int = 20000, seed: int = C.SEED) -> float:
    """Expected cosine similarity of two random (distinct, non-zero) word types.
    ~0 for isotropic vectors; large positive = all vectors in a narrow cone."""
    rng = np.random.default_rng(seed)
    V = np.asarray(V, dtype=np.float64)
    idx = np.flatnonzero(np.linalg.norm(V, axis=1) > 0)
    a, b = rng.choice(idx, n_pairs), rng.choice(idx, n_pairs)
    keep = a != b
    U = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)
    return float((U[a[keep]] * U[b[keep]]).sum(1).mean())


def l2_normalise(V: np.ndarray) -> np.ndarray:
    return (V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)).astype(np.float32)


# --------------------------------------------------------------------------- contextual (test split)
class ContextualTest:
    """In-context word vectors of test sentences, from the Task 2 feature cache."""

    def __init__(self):
        self.offsets = load_offsets("test")
        self.row = {sid: i for i, sid in enumerate(load_meta("test")["sent_ids"])}
        self._layers: dict[int, np.ndarray] = {}

    def vectors(self, s: Sentence, layer: int) -> np.ndarray:
        if layer not in self._layers:
            self._layers[layer] = load_layer("test", layer, mmap=True)
        i = self.row[s.sent_id]
        h = np.asarray(self._layers[layer][self.offsets[i]:self.offsets[i + 1]], dtype=np.float64)
        assert len(h) == len(s), f"feature/token mismatch in {s.sent_id}"
        return h


# --------------------------------------------------------------------------- Task 2 probe
def probe_matrix(layer: int, rank: int, probe_dir: Path | None = None) -> np.ndarray:
    """B [768, k] of the trained distance probe."""
    probe_dir = probe_dir or (C.TASK2_OUT / "probes")
    ck = torch.load(probe_dir / f"L{layer}_distance_k{rank}.pt", map_location="cpu")
    return ck["state_dict"]["proj"].double().numpy()


def probe_sq_dist(h: np.ndarray, B: np.ndarray) -> np.ndarray:
    """||B^T (h_i - h_j)||^2 for all pairs (the probe's predicted tree distance)."""
    t = h @ B
    sq = (t * t).sum(1)
    return np.maximum(sq[:, None] + sq[None, :] - 2.0 * t @ t.T, 0.0)


def euclidean_dist(h: np.ndarray) -> np.ndarray:
    sq = (h * h).sum(1)
    return np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2.0 * h @ h.T, 0.0))


In [ ]:
if MODE == "pretrained":   # optional shortcut: our trained checkpoints from the project repository
    import urllib.request
    os.makedirs("outputs/task2/probes", exist_ok=True)
    base = "https://raw.githubusercontent.com/trishitasaha/NLP_Assignment2/main/23CS10081_Yellapu_Dheeraj_Babu/outputs/task2/probes"
    for layer in range(13):
        for probe in ("distance", "depth"):
            for k in (32, 64, 128):
                name = f"L{layer}_{probe}_k{k}.pt"
                if not os.path.exists(f"outputs/task2/probes/{name}"):
                    urllib.request.urlretrieve(f"{base}/{name}", f"outputs/task2/probes/{name}")
    print("downloaded 78 trained probes; training below will skip them")

In [ ]:
flags = ["--tasks", "2"] + (["--fast"] if MODE == "fast" else []) + (["--with-control"] if WITH_CONTROL else [])
run("code/run_all.py", *flags)

## 5. Task 3 — BERT-derived static vectors (`code/task3/`)
Average the layer-ℓ* vectors of every training occurrence per word type → the same evaluation as Task 1, with paired
significance tests; plus the layer sweep, the all-but-the-top fix, the diagnostic ladder and the diagnostics.

In [ ]:
%%writefile code/task3/diagnostics_task3.py
"""Task 3 diagnostics: turn the three explanations of "T3 wins on meaning, loses on trees" into
measured numbers (plan.md Phase 4 caveat 1).

A. HUBNESS (Radovanovic et al., JMLR 2010). N_10(w) = how often w appears in the 10-nearest-
   neighbour lists of the other words. A long right tail (high skewness) means a few "hub"
   words are close to everything. Reported: skewness of N_10, the top hubs and how many of the
   top-50 hubs are function words. Vocabulary: word types with train frequency >= 5 (rare-word
   vectors are noise for both models).

B. HUBS IN THE MSTs (660-sentence pool). Mean degree of function-word tokens in the predicted
   trees vs the gold trees, and the largest node degree per sentence ("star" trees).

C. PARADIGMATIC vs SYNTAGMATIC (Sahlgren 2006).
   C1 nearest neighbours: share of each word's top-10 neighbours with the same majority UPOS
      (paradigmatic: substitutes share a category), and share that actually co-occur with it
      within +-5 words in train (syntagmatic: partners), each vs a random-pair baseline.
   C2 trees: P(same UPOS) for gold edges vs non-edge word pairs in the same sentence, and for the
      edges each representation predicts. If gold edges mostly join *different* categories,
      a similarity that prefers same-category pairs is the wrong signal for trees.
   C3 edge AUC: within each sentence, the probability that a gold edge pair is closer than a
      non-edge pair (0.5 = no information). Measures "does closeness predict attachment",
      independently of the MST algorithm.

Usage (from the project root, after evaluate_task3.py):
    python code/task3/diagnostics_task3.py
Outputs -> outputs/task3/diagnostics/: diagnostics.json, hubs.csv, edge_stats.csv
"""
from __future__ import annotations

import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, skew

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "task1"))
from common import config as C  # noqa: E402
from common.data import load_ewt, word_counts  # noqa: E402
from common.static_eval import _sentence_vectors, get_mst_sentences  # noqa: E402
from common.trees import cosine_distance_matrix, gold_mst_edges, linear_chain_dist, mst_on_subset  # noqa: E402

from build_ppmi_svd import cooccurrence_matrix  # noqa: E402
from t3_common import (ContextualTest, abtt, best_layer, load_t1, load_t3_static, load_w2i,  # noqa: E402
                       probe_matrix, probe_sq_dist)

OUT = C.TASK3_OUT / "diagnostics"
FUNCTION_UPOS = {"DET", "ADP", "PRON", "AUX", "CCONJ", "SCONJ", "PART"}
K = 10
MIN_FREQ_HUB = 5
MIN_FREQ_NN = 10


def majority_upos(train) -> dict[str, str]:
    c = defaultdict(Counter)
    for s in train:
        for w, u in zip(s.lower, s.upos):
            c[w][u] += 1
    return {w: cnt.most_common(1)[0][0] for w, cnt in c.items()}


def unit(V):
    return V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)


def knn(U: np.ndarray, k: int = K, chunk: int = 2048) -> np.ndarray:
    """Indices of the k nearest neighbours (cosine) of every row, excluding itself."""
    out = np.empty((len(U), k), dtype=np.int64)
    for s in range(0, len(U), chunk):
        S = U[s:s + chunk] @ U.T
        S[np.arange(len(S)), np.arange(s, s + len(S))] = -np.inf
        idx = np.argpartition(-S, k, axis=1)[:, :k]
        order = np.argsort(-np.take_along_axis(S, idx, 1), axis=1)
        out[s:s + chunk] = np.take_along_axis(idx, order, 1)
    return out


# --------------------------------------------------------------------------- A + C1
def lexical_diagnostics(name, V, w2i, counts, upos, M, rng) -> tuple[dict, pd.DataFrame]:
    i2w = {i: w for w, i in w2i.items()}
    # A. hubness over frequent-enough types
    hub_ids = np.array(sorted(i for w, i in w2i.items() if counts[w] >= MIN_FREQ_HUB and np.any(V[i])))
    nn = knn(unit(V[hub_ids].astype(np.float64)))
    Nk = np.bincount(nn.ravel(), minlength=len(hub_ids))
    top = np.argsort(-Nk)[:50]
    hubs = pd.DataFrame(dict(system=name, rank=np.arange(1, 51), word=[i2w[int(hub_ids[t])] for t in top],
                             N10=Nk[top], upos=[upos.get(i2w[int(hub_ids[t])], "?") for t in top]))
    frac_fn_top50 = float(hubs.upos.isin(FUNCTION_UPOS).mean())
    # C1. neighbour category agreement and co-occurrence, among types with freq >= MIN_FREQ_NN
    ids = np.array(sorted(i for w, i in w2i.items() if counts[w] >= MIN_FREQ_NN and np.any(V[i])))
    nn = ids[knn(unit(V[ids].astype(np.float64)))]
    up = np.array([upos[i2w[int(i)]] for i in ids])
    up_of = dict(zip(ids.tolist(), up.tolist()))
    same = np.array([[up_of[int(j)] == up_of[int(q)] for j in row] for q, row in zip(ids, nn)])
    Mb = (M[ids][:, ids] > 0).tocsr()
    pos = {int(i): p for p, i in enumerate(ids)}
    cooc = np.array([[Mb[pos[int(q)], pos[int(j)]] for j in row] for q, row in zip(ids, nn)])
    ra, rb = rng.choice(len(ids), 200000), rng.choice(len(ids), 200000)
    keep = ra != rb
    ra, rb = ra[keep], rb[keep]
    base_same = float((up[ra] == up[rb]).mean())
    base_cooc = float(np.asarray(Mb[ra, rb]).ravel().mean())
    res = dict(system=name, n_hub_vocab=int(len(hub_ids)), N10_skewness=float(skew(Nk)), N10_max=int(Nk.max()),
               frac_function_words_top50_hubs=frac_fn_top50, top10_hubs=hubs.word.head(10).tolist(),
               n_nn_vocab=int(len(ids)), nn_same_upos=float(same.mean()), nn_cooccur=float(cooc.mean()),
               random_same_upos=base_same, random_cooccur=base_cooc)
    print(f"   {name:16s} N10 skew={res['N10_skewness']:.2f} max={res['N10_max']} fn-hubs(top50)={frac_fn_top50:.2f} "
          f"top hubs={res['top10_hubs'][:6]} | NN same-UPOS={res['nn_same_upos']:.3f} (random {base_same:.3f}) "
          f"NN co-occur={res['nn_cooccur']:.3f} (random {base_cooc:.3f})", flush=True)
    return res, hubs


# --------------------------------------------------------------------------- B + C2 + C3
def tree_diagnostics(pool, dist_fns: dict) -> pd.DataFrame:
    stats = {k: defaultdict(float) for k in ["gold", *dist_fns]}
    for s in pool:
        keep = s.nonpunct_mask
        idx = np.flatnonzero(keep)
        is_fn = np.array([u in FUNCTION_UPOS for u in s.upos])
        gold = gold_mst_edges(s.heads, keep)
        pairs = [(int(a), int(b)) for x, a in enumerate(idx) for b in idx[x + 1:]]
        non = [p for p in pairs if p not in gold]
        for key in stats:
            if key == "gold":
                edges, D = gold, None
            else:
                D = dist_fns[key](s)
                edges = mst_on_subset(D, keep)
            deg = Counter([a for a, _ in edges] + [b for _, b in edges])
            st = stats[key]
            st["n_sent"] += 1
            st["edges"] += len(edges)
            st["fn_tokens"] += int(is_fn[idx].sum())
            st["fn_degree_sum"] += sum(deg[int(i)] for i in idx if is_fn[i])
            st["edges_touching_fn"] += sum(is_fn[a] or is_fn[b] for a, b in edges)
            st["max_degree_sum"] += max(deg.values()) if deg else 0
            st["same_upos_edges"] += sum(s.upos[a] == s.upos[b] for a, b in edges)
            if key == "gold":
                st["nonedge_pairs"] += len(non)
                st["same_upos_nonedges"] += sum(s.upos[a] == s.upos[b] for a, b in non)
            elif non and gold:  # C3: AUC of -distance for gold edges vs non-edges (ties = 0.5)
                sc = np.array([-D[a, b] for a, b in list(gold) + non])
                r = rankdata(sc)
                n1, n0 = len(gold), len(non)
                st["auc_sum"] += (r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
                st["auc_n"] += 1
    rows = []
    for key, st in stats.items():
        rows.append(dict(
            system=key, mean_fn_token_degree=st["fn_degree_sum"] / max(st["fn_tokens"], 1),
            share_edges_touching_fn=st["edges_touching_fn"] / st["edges"],
            mean_max_degree=st["max_degree_sum"] / st["n_sent"],
            p_same_upos_edges=st["same_upos_edges"] / st["edges"],
            p_same_upos_nonedges=st["same_upos_nonedges"] / st["nonedge_pairs"] if key == "gold" else np.nan,
            edge_auc=st["auc_sum"] / st["auc_n"] if st["auc_n"] else np.nan))
    return pd.DataFrame(rows)


def main() -> None:
    C.set_seed()
    OUT.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(C.SEED)
    t0 = time.time()
    bl = best_layer()
    L, k = bl["l_star"], bl["distance_rank"]
    w2i = load_w2i()
    train = load_ewt("train")
    counts = word_counts(train)
    upos = majority_upos(train)
    M = cooccurrence_matrix(train, w2i, C.WINDOW)
    T1, T3 = load_t1(), load_t3_static(L)
    T3a = abtt(T3)

    print("[diag] A + C1: hubness and nearest-neighbour types", flush=True)
    lex, hub_tabs = [], []
    for name, V in (("T1 PPMI-SVD", T1), (f"T3 L{L}", T3), (f"T3 L{L} + ABTT", T3a), ("T3 L0", load_t3_static(0))):
        r, h = lexical_diagnostics(name, V, w2i, counts, upos, M, rng)
        lex.append(r)
        hub_tabs.append(h)
    pd.concat(hub_tabs).to_csv(OUT / "hubs.csv", index=False)

    print("[diag] B + C2 + C3: tree statistics on the 660-sentence pool", flush=True)
    _, pool = get_mst_sentences()
    ctx, Bm = ContextualTest(), probe_matrix(L, k)

    def static(V):
        return lambda s: cosine_distance_matrix(_sentence_vectors(s, V, w2i))

    fns = {"linear chain": lambda s: linear_chain_dist(len(s)), "T1 PPMI-SVD": static(T1),
           f"T3 L{L}": static(T3), f"T3 L{L} + ABTT": static(T3a),
           f"contextual L{L} cosine": lambda s: cosine_distance_matrix(ctx.vectors(s, L)),
           f"T2 probe L{L}": lambda s: probe_sq_dist(ctx.vectors(s, L), Bm)}
    trees = tree_diagnostics(pool, fns)
    trees.to_csv(OUT / "edge_stats.csv", index=False)

    (OUT / "diagnostics.json").write_text(json.dumps(dict(lexical=lex, trees=trees.to_dict("records"),
                                                          function_upos=sorted(FUNCTION_UPOS)), indent=1))
    print("\n================ TASK 3 DIAGNOSTICS ================")
    with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
        print(pd.DataFrame(lex).drop(columns=["top10_hubs"]).to_string(index=False))
        print()
        print(trees.to_string(index=False))
    print(f"[diag] done in {(time.time() - t0) / 60:.1f} min -> {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task3/evaluate_task3.py
"""Task 3b: evaluate the BERT-static vectors with the *identical* Task 1 protocol and compare.

What it runs (everything reuses common/static_eval.evaluate_static_vectors, the very function
Task 1 used, so the protocol is identical by construction):
  * T3   = BERT-static vectors from layer l* (dev-selected in Task 2), mean-pooled over train
  * T3+  = the same vectors after all-but-the-top (mean removal + top-7 PCs removed), the
           Tier B anisotropy fix
  * T2 reference row: the trained Task 2 distance probe (layer l*) on the same 20 sentences and
    the same 660-sentence pool (contextual, supervised; an upper reference, not a competitor)
  * paired significance tests against T1 on the *same items*:
        WordSim-353  paired bootstrap of the Spearman difference (same covered pairs)
        analogy      exact McNemar test (same covered questions)
        MST          paired bootstrap of the UUAS difference (same sentences)

Usage (from the project root, after pool_static.py):
    python code/task3/evaluate_task3.py

Outputs -> outputs/task3/
    main/, main_abtt/         full per-benchmark outputs (same files as outputs/task1/)
    t2_reference_mst_20.json  the probe's trees on the 20 sentences
    comparison_table.csv      T1 spec, T1 best variant (p0), T3, T3+ABTT, T2 reference
    paired_tests.json         the significance tests
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import benchmarks as B  # noqa: E402
from common import config as C  # noqa: E402
from common.static_eval import (evaluate_static_vectors, get_mst_sentences, mst_check,  # noqa: E402
                                per_sentence_frame, static_mst_check)
from common.stats import mcnemar_exact, paired_bootstrap_spearman, paired_bootstrap_uuas  # noqa: E402

from t3_common import (ContextualTest, abtt, best_layer, load_t1, load_t3_static, load_w2i,  # noqa: E402
                       probe_matrix, probe_sq_dist, t1_dir)

T1_BEST_TAG = "p0"  # best classical variant found in the Phase 1 ablations (eigenvalue weight p = 0)


def table_row(system: str, r: dict) -> dict:
    ws, an, m20, mp = r["wordsim"], r["analogy"], r["mst_20"], r.get("mst_pool") or {}
    return dict(
        system=system,
        ws_rho_cov=ws["spearman_covered"], ws_ci_low=ws["spearman_covered_ci95"][0],
        ws_ci_high=ws["spearman_covered_ci95"][1], ws_n=ws["n_covered"], ws_rho_full=ws["spearman_full"],
        ws_rho_freq20=ws.get("by_min_train_freq", {}).get("20", {}).get("spearman"),
        an_acc_cov=an["acc_covered"], an_sem_cov=an["by_kind"]["semantic"]["acc_covered"],
        an_syn_cov=an["by_kind"]["syntactic"]["acc_covered"], an_acc_full=an["acc_full"], an_n=an["n_covered"],
        mst20_uuas=m20["uuas"], mst20_ci_low=m20["uuas_ci95"][0], mst20_ci_high=m20["uuas_ci95"][1],
        pool_uuas=mp.get("uuas"))


def ws_pairs(d: Path) -> pd.DataFrame:
    """All 353 rows in benchmark order (WS-353 contains one duplicated pair, so rows are
    aligned by position, not merged on the words)."""
    return pd.read_csv(d / "wordsim_pairs.csv")[["word1", "word2", "human", "model", "covered"]]


def analogy_hits(examples: pd.DataFrame) -> pd.DataFrame:
    return examples[["section", "a", "b", "c", "gold", "correct"]]


def pool_frame(d: Path, vectors, w2i, pool) -> pd.DataFrame:
    """Per-sentence pool counts; recomputed if an older run did not save them."""
    f = d / "mst_pool_per_sentence.csv"
    if f.exists():
        return pd.read_csv(f)
    return per_sentence_frame(static_mst_check(pool, vectors, w2i, "pool", n_random=0))


def paired_block(name_a, name_b, A: dict, Bd: dict) -> dict:
    """A, Bd: dict(ws=DataFrame, an=DataFrame, m20=DataFrame, pool=DataFrame)."""
    out = dict(a=name_a, b=name_b)
    if A.get("ws") is not None and Bd.get("ws") is not None:
        a, b = A["ws"], Bd["ws"]
        assert (a.word1.values == b.word1.values).all() and (a.word2.values == b.word2.values).all()
        both = (a.covered & b.covered).values
        out["wordsim"] = dict(n=int(both.sum()), **paired_bootstrap_spearman(
            a.model.values[both], b.model.values[both], a.human.values[both]))
    if A.get("an") is not None and Bd.get("an") is not None:
        a = A["an"].merge(Bd["an"], on=["section", "a", "b", "c", "gold"], suffixes=("_a", "_b"))
        out["analogy"] = dict(n=len(a), acc_a=float(a.correct_a.mean()), acc_b=float(a.correct_b.mean()),
                              **mcnemar_exact(a.correct_a, a.correct_b))
    for key in ("m20", "pool"):
        m = A[key].merge(Bd[key], on=["sent_id", "total"], suffixes=("_a", "_b"))
        out["mst_20" if key == "m20" else "mst_pool"] = dict(
            n_sentences=len(m), **paired_bootstrap_uuas(m.correct_a, m.correct_b, m.total))
    return out


def main() -> None:
    C.set_seed()
    bl = best_layer()
    L, k = bl["l_star"], bl["distance_rank"]
    w2i = load_w2i()
    sample, pool = get_mst_sentences()
    print(f"[t3] l* = {L} (distance-probe rank {k}); vocab = {len(w2i)}", flush=True)

    # ---- T3 (required) and T3 + all-but-the-top (Tier B)
    V3 = load_t3_static(L)
    r3 = evaluate_static_vectors(V3, w2i, f"T3 BERT-static (layer {L}, mean-pooled)", C.TASK3_OUT / "main")
    r3a = evaluate_static_vectors(abtt(V3), w2i, f"T3 BERT-static (layer {L}) + ABTT(D=7)",
                                  C.TASK3_OUT / "main_abtt")

    # ---- T2 reference: the probe on the same sentences
    ctx, Bm = ContextualTest(), probe_matrix(L, k)
    fn = lambda s: probe_sq_dist(ctx.vectors(s, L), Bm)  # noqa: E731
    t2_20 = mst_check(sample, fn, f"T2 distance probe (layer {L}, k={k})")
    t2_pool = mst_check(pool, fn, "T2 probe pool", n_random=0)
    (C.TASK3_OUT / "t2_reference_mst_20.json").write_text(json.dumps(t2_20, indent=1))

    # ---- comparison table
    rows = [table_row("T1 PPMI-SVD (spec)", json.loads((t1_dir() / "results.json").read_text())),
            table_row(f"T1 PPMI-SVD best variant ({T1_BEST_TAG})",
                      json.loads((t1_dir(T1_BEST_TAG) / "results.json").read_text())),
            table_row(f"T3 BERT-static L{L}", r3), table_row(f"T3 BERT-static L{L} + ABTT", r3a)]
    rows.append(dict(system=f"T2 probe L{L} k={k} (contextual, supervised; reference)",
                     mst20_uuas=t2_20["uuas"], mst20_ci_low=t2_20["uuas_ci95"][0],
                     mst20_ci_high=t2_20["uuas_ci95"][1], pool_uuas=t2_pool["uuas"]))
    rows.append(dict(system="baseline: linear chain", mst20_uuas=t2_20["chain_uuas"], pool_uuas=t2_pool["chain_uuas"]))
    rows.append(dict(system="baseline: random tree", mst20_uuas=r3["mst_20"]["random_uuas"],
                     pool_uuas=r3["mst_pool"]["random_uuas"]))
    table = pd.DataFrame(rows)
    table.to_csv(C.TASK3_OUT / "comparison_table.csv", index=False)

    # ---- paired tests on identical items
    def per_sent_20(path: Path) -> pd.DataFrame:
        return per_sentence_frame(json.loads(path.read_text()))

    t1_examples = pd.DataFrame(B.evaluate_analogy(load_t1(), w2i)["examples"])  # all covered questions
    sys_ = {
        "T1": dict(ws=ws_pairs(t1_dir()), an=analogy_hits(t1_examples),
                   m20=per_sent_20(t1_dir() / "mst_20_sentences.json"),
                   pool=pool_frame(t1_dir(), load_t1(), w2i, pool)),
        "T3": dict(ws=ws_pairs(C.TASK3_OUT / "main"),
                   an=analogy_hits(pd.read_csv(C.TASK3_OUT / "main" / "analogy_examples.csv")),
                   m20=per_sent_20(C.TASK3_OUT / "main" / "mst_20_sentences.json"),
                   pool=pd.read_csv(C.TASK3_OUT / "main" / "mst_pool_per_sentence.csv")),
        "T3+ABTT": dict(ws=ws_pairs(C.TASK3_OUT / "main_abtt"),
                        an=analogy_hits(pd.read_csv(C.TASK3_OUT / "main_abtt" / "analogy_examples.csv")),
                        m20=per_sent_20(C.TASK3_OUT / "main_abtt" / "mst_20_sentences.json"),
                        pool=pd.read_csv(C.TASK3_OUT / "main_abtt" / "mst_pool_per_sentence.csv")),
        "T2probe": dict(m20=per_sentence_frame(t2_20), pool=per_sentence_frame(t2_pool)),
    }
    assert len(sys_["T1"]["an"]) == r3["analogy"]["n_covered"], "analogy items differ between T1 and T3"
    tests = [paired_block("T3", "T1", sys_["T3"], sys_["T1"]),
             paired_block("T3+ABTT", "T1", sys_["T3+ABTT"], sys_["T1"]),
             paired_block("T3+ABTT", "T3", sys_["T3+ABTT"], sys_["T3"]),
             paired_block("T2probe", "T3", sys_["T2probe"], sys_["T3"])]
    (C.TASK3_OUT / "paired_tests.json").write_text(json.dumps(tests, indent=1))

    # ---- summary
    print("\n================ TASK 3 vs TASK 1 (identical protocol) ================")
    with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
        print(table[["system", "ws_rho_cov", "ws_ci_low", "ws_ci_high", "ws_rho_full", "ws_rho_freq20",
                     "an_acc_cov", "an_sem_cov", "an_syn_cov", "an_acc_full", "mst20_uuas", "pool_uuas"]])
    print("\nPaired tests (A - B on identical items):")
    for t in tests:
        parts = []
        if "wordsim" in t:
            w = t["wordsim"]
            parts.append(f"WS d_rho={w['diff']:+.3f} [{w['ci_low']:+.3f},{w['ci_high']:+.3f}] (n={w['n']})")
        if "analogy" in t:
            a = t["analogy"]
            parts.append(f"analogy {a['acc_a']:.3f} vs {a['acc_b']:.3f}, McNemar {a['a_only']}/{a['b_only']} p={a['p_value']:.1e}")
        for key in ("mst_20", "mst_pool"):
            m = t[key]
            parts.append(f"{key} dUUAS={m['diff']:+.3f} [{m['ci_low']:+.3f},{m['ci_high']:+.3f}]")
        print(f"  {t['a']:8s} - {t['b']:8s}: " + " | ".join(parts))
    print("=" * 72)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task3/extensions_task3.py
"""Task 3c: analyses beyond the required comparison (plan.md Phase 4, Tier B + Tier C).

1. LAYER SWEEP (Tier B). For every BERT layer 0..12, pooled static vectors, raw and after
   all-but-the-top (ABTT): WS-353 Spearman, analogy accuracy (semantic / syntactic), MST UUAS
   (20 sentences + 660-sentence pool) and anisotropy. For context, the same pool is also parsed
   from the raw *contextual* layer vectors (cosine MST, no training) and with the Task 2
   distance probe of that layer. Question answered: is the best "syntax" layer l* also the best
   layer for static *semantics*?
   T1 is evaluated with ABTT too (D=1 for 100 dims), so any gain from ABTT is not credited to BERT.

2. DIAGNOSTIC LADDER (Tier B) on the same 20 sentences and the 660-sentence pool:
      random tree / linear chain            floor and the purely positional baseline
      T1 PPMI-SVD (spec, best variant)      classical static semantics
      T3 BERT-static l* (raw, ABTT)         BERT with the context averaged away
      raw contextual l*, cosine MST         context, but no training
      T2 distance probe l*                  context + a learned syntactic subspace
   with paired bootstrap tests between neighbouring rungs and per-relation recall.

3. TIER C variants (added to the ladder and, where meaningful, to the semantic benchmarks):
      concat   [L2(T1) ; L2(T3)]            are the two static spaces complementary?
      T3 static through the probe B         does the syntactic subspace help *static* vectors?

Usage (from the project root, after pool_static.py with all layers and evaluate_task3.py):
    python code/task3/extensions_task3.py

Outputs -> outputs/task3/extensions/
    layer_sweep.csv  ladder.csv  ladder_relations.csv  ladder_tests.json  ladder_20_edges.json
    tierC_semantic.csv
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import benchmarks as B  # noqa: E402
from common import config as C  # noqa: E402
from common.data import load_ewt, word_counts  # noqa: E402
from common.static_eval import _sentence_vectors, get_mst_sentences, mst_check, static_mst_check  # noqa: E402
from common.stats import paired_bootstrap_uuas  # noqa: E402
from common.trees import cosine_distance_matrix  # noqa: E402

from t3_common import (ContextualTest, abtt, anisotropy, best_layer, l2_normalise, load_t1,  # noqa: E402
                       load_t3_static, load_w2i, probe_matrix, probe_sq_dist)

OUT = C.TASK3_OUT / "extensions"
T1_BEST_TAG = "p0"


def semantic_scores(V, w2i, counts) -> dict:
    ws = B.evaluate_wordsim(V, w2i)
    rec = pd.DataFrame(ws["records"])
    cov = rec[rec.covered]
    minf = cov.apply(lambda r: min(counts[r.word1], counts[r.word2]), axis=1)
    an = B.evaluate_analogy(V, w2i)
    return dict(ws_rho_cov=ws["spearman_covered"], ws_rho_full=ws["spearman_full"],
                ws_rho_freq20=float(spearmanr(cov.model[minf >= 20], cov.human[minf >= 20]).statistic),
                an_acc_cov=an["acc_covered"], an_sem_cov=an["by_kind"]["semantic"]["acc_covered"],
                an_syn_cov=an["by_kind"]["syntactic"]["acc_covered"])


def static_rows(name, V, w2i, counts, sample, pool, extra=None) -> dict:
    t = time.time()
    row = dict(system=name, **(extra or {}), **semantic_scores(V, w2i, counts),
               mst20_uuas=static_mst_check(sample, V, w2i, name, n_random=0)["uuas"],
               pool_uuas=static_mst_check(pool, V, w2i, name, n_random=0)["uuas"],
               anisotropy=anisotropy(V))
    print(f"   {name:34s} WS={row['ws_rho_cov']:.3f} (f>=20 {row['ws_rho_freq20']:.3f}) "
          f"an={row['an_acc_cov']:.4f} (sem {row['an_sem_cov']:.3f} syn {row['an_syn_cov']:.3f}) "
          f"MST20={row['mst20_uuas']:.3f} pool={row['pool_uuas']:.3f} aniso={row['anisotropy']:.3f} "
          f"({time.time() - t:.0f} s)", flush=True)
    return row


# --------------------------------------------------------------------------- 1. layer sweep
def layer_sweep(w2i, counts, sample, pool, ctx, k) -> pd.DataFrame:
    print("[ext] 1. layer sweep", flush=True)
    rows = []
    for tag, label in ((None, "T1 spec"), (T1_BEST_TAG, f"T1 {T1_BEST_TAG}")):
        V = load_t1(tag)
        rows.append(static_rows(f"{label} raw", V, w2i, counts, sample, pool, dict(source="T1", layer=-1, post="raw")))
        rows.append(static_rows(f"{label} ABTT(D=1)", abtt(V), w2i, counts, sample, pool,
                                dict(source="T1", layer=-1, post="abtt")))
    for L in range(C.BERT_NUM_LAYERS + 1):
        V = load_t3_static(L)
        for post, M in (("raw", V), ("abtt", abtt(V))):
            rows.append(static_rows(f"T3 L{L} {post}", M, w2i, counts, sample, pool,
                                    dict(source="T3", layer=L, post=post)))
        # contextual references on the same pool (no static averaging)
        cos = mst_check(pool, lambda s: cosine_distance_matrix(ctx.vectors(s, L)), "ctx", n_random=0)
        Bm = probe_matrix(L, k)
        prb = mst_check(pool, lambda s: probe_sq_dist(ctx.vectors(s, L), Bm), "probe", n_random=0)
        rows.append(dict(system=f"contextual L{L} cosine", source="contextual", layer=L, post="raw",
                         pool_uuas=cos["uuas"]))
        rows.append(dict(system=f"T2 probe L{L} k={k}", source="probe", layer=L, post="probe",
                         pool_uuas=prb["uuas"]))
        print(f"   contextual L{L}: raw cosine pool={cos['uuas']:.3f}  probe pool={prb['uuas']:.3f}", flush=True)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- 2+3. ladder
def ladder(w2i, sample, pool, ctx, L, k):
    print("[ext] 2. diagnostic ladder", flush=True)
    T1, T1b = load_t1(), load_t1(T1_BEST_TAG)
    T3 = load_t3_static(L)
    T3a = abtt(T3)
    concat = np.concatenate([l2_normalise(T1), l2_normalise(T3)], axis=1)
    Bm = probe_matrix(L, k)

    def static_fn(V):
        return lambda s: cosine_distance_matrix(_sentence_vectors(s, V, w2i))

    rungs = [  # (key, label, tier, dist_fn)
        ("T1", "T1 PPMI-SVD (spec)", "required", static_fn(T1)),
        ("T1best", f"T1 PPMI-SVD ({T1_BEST_TAG})", "B", static_fn(T1b)),
        ("T3", f"T3 BERT-static L{L}", "required", static_fn(T3)),
        ("T3abtt", f"T3 BERT-static L{L} + ABTT", "B", static_fn(T3a)),
        ("concat", "concat [T1 ; T3] (L2-normalised)", "C", static_fn(concat)),
        ("T3probe", f"T3 BERT-static L{L} through probe B", "C",
         lambda s: probe_sq_dist(_sentence_vectors(s, T3, w2i), Bm)),
        ("ctx", f"contextual L{L}, cosine (no training)", "B",
         lambda s: cosine_distance_matrix(ctx.vectors(s, L))),
        ("probe", f"T2 distance probe L{L} k={k}", "reference",
         lambda s: probe_sq_dist(ctx.vectors(s, L), Bm)),
    ]
    res = {}
    for key, label, tier, fn in rungs:
        t = time.time()
        m20 = mst_check(sample, fn, label)  # 20 sentences: with the random-tree baseline
        mp = mst_check(pool, fn, label, n_random=0)
        res[key] = dict(label=label, tier=tier, m20=m20, pool=mp)
        print(f"   {label:42s} MST20={m20['uuas']:.3f} [{m20['uuas_ci95'][0]:.3f},{m20['uuas_ci95'][1]:.3f}] "
              f"pool={mp['uuas']:.3f} [{mp['uuas_ci95'][0]:.3f},{mp['uuas_ci95'][1]:.3f}] ({time.time() - t:.0f} s)",
              flush=True)

    any_ = next(iter(res.values()))
    rows = [dict(key="random", system="random tree (expected)", tier="baseline",
                 mst20_uuas=any_["m20"]["random_uuas"]),
            dict(key="chain", system="linear chain", tier="baseline", mst20_uuas=any_["m20"]["chain_uuas"],
                 mst20_ci_low=any_["m20"]["chain_ci95"][0], mst20_ci_high=any_["m20"]["chain_ci95"][1],
                 pool_uuas=any_["pool"]["chain_uuas"], pool_ci_low=any_["pool"]["chain_ci95"][0],
                 pool_ci_high=any_["pool"]["chain_ci95"][1])]
    for key, r in res.items():
        rows.append(dict(key=key, system=r["label"], tier=r["tier"],
                         mst20_uuas=r["m20"]["uuas"], mst20_ci_low=r["m20"]["uuas_ci95"][0],
                         mst20_ci_high=r["m20"]["uuas_ci95"][1], pool_uuas=r["pool"]["uuas"],
                         pool_ci_low=r["pool"]["uuas_ci95"][0], pool_ci_high=r["pool"]["uuas_ci95"][1],
                         mst20_vs_chain=r["m20"]["vs_chain"]["diff"], pool_vs_chain=r["pool"]["vs_chain"]["diff"]))

    # paired tests between rungs, on identical sentences
    def arr(m):
        return (np.array([p["correct"] for p in m["per_sentence"]]),
                np.array([p["total"] for p in m["per_sentence"]]))

    pairs = [("T3", "T1"), ("T3abtt", "T1"), ("T3abtt", "T3"), ("T1best", "T1"), ("concat", "T3"),
             ("T3probe", "T3"), ("ctx", "T3"), ("probe", "ctx"), ("probe", "T3")]
    tests = []
    for a, b in pairs:
        t = dict(a=res[a]["label"], b=res[b]["label"])
        for part in ("m20", "pool"):
            ca, tot = arr(res[a][part])
            cb, _ = arr(res[b][part])
            t[part] = paired_bootstrap_uuas(ca, cb, tot)
        tests.append(t)

    rel_rows = [dict(key=key, system=r["label"], relation=rel, **v)
                for key, r in res.items() for rel, v in r["pool"]["per_relation"].items()]
    edges20 = {key: [dict(sent_id=p["sent_id"], correct=p["correct"], total=p["total"],
                          pred_edges=p["pred_edges"]) for p in r["m20"]["per_sentence"]] for key, r in res.items()}
    edges20["gold"] = [dict(sent_id=p["sent_id"], tokens=p["tokens"], text=p["text"], gold_edges=p["gold_edges"])
                       for p in any_["m20"]["per_sentence"]]
    return pd.DataFrame(rows), pd.DataFrame(rel_rows), tests, edges20, concat


def main() -> None:
    C.set_seed()
    OUT.mkdir(parents=True, exist_ok=True)
    bl = best_layer()
    L, k = bl["l_star"], bl["distance_rank"]
    w2i = load_w2i()
    counts = word_counts(load_ewt("train"))
    sample, pool = get_mst_sentences()
    ctx = ContextualTest()
    t0 = time.time()

    lad, rel, tests, edges20, concat = ladder(w2i, sample, pool, ctx, L, k)
    lad.to_csv(OUT / "ladder.csv", index=False)
    rel.to_csv(OUT / "ladder_relations.csv", index=False)
    (OUT / "ladder_tests.json").write_text(json.dumps(tests, indent=1))
    (OUT / "ladder_20_edges.json").write_text(json.dumps(edges20, indent=1))

    print("[ext] 3. Tier C on the semantic benchmarks", flush=True)
    tc = pd.DataFrame([static_rows("concat [T1 ; T3]", concat, w2i, counts, sample, pool)])
    tc.to_csv(OUT / "tierC_semantic.csv", index=False)

    sweep = layer_sweep(w2i, counts, sample, pool, ctx, k)
    sweep.to_csv(OUT / "layer_sweep.csv", index=False)

    print("\n================ TASK 3 EXTENSIONS ================")
    with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
        print(lad[["system", "tier", "mst20_uuas", "mst20_ci_low", "mst20_ci_high", "pool_uuas", "pool_ci_low",
                   "pool_ci_high"]].to_string(index=False))
        print("\nPaired tests (A - B, same sentences):")
        for t in tests:
            print(f"  {t['a'][:34]:34s} - {t['b'][:28]:28s}: "
                  f"20: {t['m20']['diff']:+.3f} [{t['m20']['ci_low']:+.3f},{t['m20']['ci_high']:+.3f}]  "
                  f"pool: {t['pool']['diff']:+.3f} [{t['pool']['ci_low']:+.3f},{t['pool']['ci_high']:+.3f}]")
        st = sweep[sweep.source == "T3"]
        for post in ("raw", "abtt"):
            s = st[st.post == post]
            print(f"\nT3 {post}: best layer WS={int(s.loc[s.ws_rho_cov.idxmax(), 'layer'])} "
                  f"({s.ws_rho_cov.max():.3f}), analogy={int(s.loc[s.an_acc_cov.idxmax(), 'layer'])} "
                  f"({s.an_acc_cov.max():.3f}), pool MST={int(s.loc[s.pool_uuas.idxmax(), 'layer'])} "
                  f"({s.pool_uuas.max():.3f}); l*={L}")
    print(f"[ext] done in {(time.time() - t0) / 60:.1f} min -> {OUT}")
    print("=" * 72)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task3/plot_task3.py
"""Task 3 figures + qualitative tables (same style/palette as the Task 2 figures).

Usage (from the project root, after evaluate_task3.py and extensions_task3.py):
    python code/task3/plot_task3.py

Figures -> outputs/task3/figures/ (PNG for slides, PDF for the report):
  layer_sweep        2x2: WS-353 rho, analogy acc, MST UUAS (pool), anisotropy vs BERT layer
  ladder             MST UUAS of every representation on the 660-sentence pool (95% CI) + the 20 sentences
  ladder_relations   recall per dependency relation x representation (pool)
  ws_by_frequency    WS-353 rho restricted to pairs whose words occur >= f times in train
  mst_example        one of the 20 sentences: gold tree vs T1 / T3 / contextual / probe MSTs
Tables -> outputs/task3/qualitative/:
  nearest_neighbours.csv   T1 vs T3 neighbours of the probe words
  analogy_examples.csv     questions solved by one model and not the other
  mst20_per_sentence.csv   UUAS of each representation on each of the 20 sentences
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "task2"))
from common import config as C  # noqa: E402

import plot_task2 as P2  # noqa: E402  (palette, rcParams, save(), arc_diagram())
from plot_task2 import GRID, INK, INK2, MARKERS, SEQ_CMAP, SERIES, plt  # noqa: E402,F401

from t3_common import best_layer, t1_dir  # noqa: E402

P2.OUT = C.TASK3_OUT / "figures"
EXT = C.TASK3_OUT / "extensions"
QUAL = C.TASK3_OUT / "qualitative"


def ref_line(ax, y, text, style="--", right=False):
    """Horizontal reference line, labelled at the left (or right) end so two labels never collide."""
    ax.axhline(y, color=INK2, linestyle=style, linewidth=1.0)
    ax.annotate(text, (12 if right else 0, y), xytext=(-2 if right else 2, 3), textcoords="offset points",
                color=INK2, fontsize=8, ha="right" if right else "left")


def fig_layer_sweep(sw: pd.DataFrame, L: int) -> None:
    t3 = sw[sw.source == "T3"]
    t1 = sw[(sw.source == "T1") & sw.system.str.startswith("T1 spec")].set_index("post")
    ctx = sw[sw.source == "contextual"].sort_values("layer")
    prb = sw[sw.source == "probe"].sort_values("layer")
    fig, axes = plt.subplots(2, 2, figsize=(10, 6.6), sharex=True)
    panels = (("ws_rho_cov", "WordSim-353 Spearman ρ (covered pairs)", 1),
              ("an_acc_cov", "Analogy accuracy, covered (%)", 100),
              ("pool_uuas", "MST UUAS, 660 test sentences (%)", 100),
              ("anisotropy", "Anisotropy (mean cos of random word pairs)", 1))
    for ax, (col, title, sc) in zip(axes.ravel(), panels):
        for i, (post, lab) in enumerate((("raw", "T3 BERT-static (mean-pooled)"), ("abtt", "T3 + all-but-the-top"))):
            s = t3[t3.post == post].sort_values("layer")
            ax.plot(s.layer, s[col] * sc, color=SERIES[i], marker=MARKERS[i], markersize=5, label=lab)
        if col == "pool_uuas":
            ax.plot(ctx.layer, ctx[col] * sc, color=SERIES[2], marker=MARKERS[2], markersize=5,
                    label="contextual vectors, cosine (no training)")
            ax.plot(prb.layer, prb[col] * sc, color=INK, linewidth=1.0, marker=".", label="T2 probe (supervised)")
        ref_line(ax, t1.loc["raw", col] * sc, "T1 PPMI-SVD", right=(col == "anisotropy"))
        if col != "anisotropy":
            ref_line(ax, t1.loc["abtt", col] * sc, "T1 + ABTT", ":", right=True)
        lo, hi = ax.get_ylim()
        hi = hi + 0.10 * (hi - lo)
        ax.set_ylim(lo, hi)
        ax.axvline(L, color=INK2, linestyle=":", linewidth=1.0)
        ax.annotate(f"ℓ* = {L}", (L, hi), xytext=(4, -10), textcoords="offset points", color=INK2, fontsize=8)
        ax.set_title(title, loc="left", fontsize=10)
        ax.set_xticks(range(0, 13))
        ax.set_xlabel("BERT layer ℓ")
    h1, l1 = axes[0, 0].get_legend_handles_labels()
    h2, l2 = axes[1, 0].get_legend_handles_labels()
    fig.legend(h1 + h2[2:], l1 + l2[2:], loc="upper right", ncol=2, fontsize=8, bbox_to_anchor=(1.0, 1.0))
    fig.suptitle("Static vectors pooled from each BERT layer vs. T1 (dashed/dotted = T1 references)",
                 x=0.01, ha="left", y=1.04)
    fig.tight_layout()
    P2.save(fig, "layer_sweep")


def fig_ladder(lad: pd.DataFrame) -> None:
    lad = lad.iloc[::-1].reset_index(drop=True)  # first rung at the top
    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    y = np.arange(len(lad))
    has = lad.pool_uuas.notna()
    err = np.vstack([(lad.pool_uuas - lad.pool_ci_low)[has], (lad.pool_ci_high - lad.pool_uuas)[has]]) * 100
    ax.errorbar(lad.pool_uuas[has] * 100, y[has], xerr=err, fmt="o", color=SERIES[0], markersize=7,
                elinewidth=1.4, capsize=3, label="660-sentence pool (95% CI)")
    ax.plot(lad.mst20_uuas * 100, y, linestyle="none", marker="s", markersize=6, markerfacecolor="none",
            markeredgecolor=SERIES[1], markeredgewidth=1.4, label="the 20 sampled sentences")
    for yi, v in zip(y, lad.pool_uuas.fillna(lad.mst20_uuas)):
        ax.annotate(f"{v * 100:.1f}", (v * 100, yi), xytext=(8, 5), textcoords="offset points", fontsize=8, color=INK2)
    ax.set_yticks(y, lad.system)
    ax.set_xlabel("UUAS of the MST (%)")
    ax.set_xlim(0, 100)
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title("Structural check: how much of the dependency tree each representation recovers", loc="left", fontsize=10)
    P2.save(fig, "ladder")


def fig_ladder_relations(rel: pd.DataFrame) -> None:
    keys = ["T1", "T3", "T3abtt", "ctx", "probe"]
    rel = rel[rel.key.isin(keys)]
    top = rel[rel.key == "T1"].sort_values("total", ascending=False).relation.head(14).tolist()
    M = np.array([[rel[(rel.key == k) & (rel.relation == r)].recall.iloc[0] for k in keys] for r in top])
    names = ["T1\nPPMI-SVD", "T3\nBERT-static", "T3\n+ ABTT", "contextual\ncosine", "T2\nprobe"]
    fig, ax = plt.subplots(figsize=(8.4, 5.6))
    im = ax.imshow(M, cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.2f}".lstrip("0"), ha="center", va="center", fontsize=8,
                    color="white" if M[i, j] > 0.6 else INK)
    ax.set_xticks(range(len(keys)), names, fontsize=8)
    n_tot = {r: int(rel[(rel.key == "T1") & (rel.relation == r)].total.iloc[0]) for r in top}
    ax.set_yticks(range(len(top)), [f"{r} (n={n_tot[r]})" for r in top])
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="recall of gold edges")
    ax.set_title("Which dependency relations each representation recovers (660 test sentences)", loc="left", fontsize=10)
    P2.save(fig, "ladder_relations")


def fig_ws_by_frequency(results: dict[str, dict]) -> None:
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for i, (name, r) in enumerate(results.items()):
        bf = r["wordsim"]["by_min_train_freq"]
        th = [t for t in bf if bf[t]["n"] >= 10]
        ax.plot([int(t) for t in th], [bf[t]["spearman"] for t in th], color=SERIES[i], marker=MARKERS[i],
                markersize=5, label=name)
        if i == 0:
            for t in th:
                ax.annotate(f"n={bf[t]['n']}", (int(t), bf[t]["spearman"]), xytext=(0, 8),
                            textcoords="offset points", fontsize=7, color=INK2, ha="center")
    ax.set_xscale("log")
    ax.set_xticks([1, 5, 10, 20, 50], ["≥1", "≥5", "≥10", "≥20", "≥50"])
    ax.set_xlabel("both words occur at least f times in EWT train")
    ax.set_ylabel("WordSim-353 Spearman ρ")
    ax.legend(loc="best", fontsize=8)
    ax.minorticks_off()
    ax.set_title("WordSim-353 ρ on pairs whose words are at least this frequent", loc="left", fontsize=10)
    P2.save(fig, "ws_by_frequency")


def fig_mst_example(edges: dict) -> dict:
    gold = edges["gold"]
    keys = [("T1", "T1 PPMI-SVD"), ("T3", "T3 BERT-static"), ("ctx", "contextual, cosine"), ("probe", "T2 probe")]
    rows = []
    for i, g in enumerate(gold):
        sc = {k: edges[k][i]["correct"] / edges[k][i]["total"] for k, _ in keys}
        rows.append(dict(i=i, sent_id=g["sent_id"], n=edges["T1"][i]["total"] + 1, text=g["text"], **sc))
    df = pd.DataFrame(rows)
    df.drop(columns="i").to_csv(QUAL / "mst20_per_sentence.csv", index=False)
    cand = df[(df.n >= 8) & (df.n <= 14)]
    cand = cand if len(cand) else df
    # a *representative* sentence (not a cherry-picked one): each method's UUAS on it is as close
    # as possible to that method's average over the 20 sentences
    dev = sum((cand[k] - df[k].mean()).abs() for k, _ in keys)
    pick = cand.assign(dev=dev).sort_values(["dev", "n"]).iloc[0]
    i = int(pick.i)
    toks = gold[i]["tokens"]
    g_edges = {tuple(e) for e in gold[i]["gold_edges"]}
    fig, axes = plt.subplots(len(keys), 1, figsize=(10, 2.3 * len(keys)))
    for ax, (k, lab) in zip(axes, keys):
        pred = {tuple(e) for e in edges[k][i]["pred_edges"]}
        P2.arc_diagram(ax, toks, g_edges, pred, f"{lab}: UUAS {len(pred & g_edges) / len(g_edges):.2f}",
                       pred_label=f"{lab} MST")
    fig.suptitle(f"A representative one of the 20 sentences ({gold[i]['sent_id']}): gold tree above, MST of each representation below",
                 x=0.01, ha="left", y=1.0)
    fig.tight_layout()
    P2.save(fig, "mst_example")
    return dict(sent_id=gold[i]["sent_id"], text=gold[i]["text"], **{k: float(pick[k]) for k, _ in keys})


def tables(L: int) -> None:
    nn1 = json.loads((t1_dir() / "nearest_neighbours.json").read_text())
    nn3 = json.loads((C.TASK3_OUT / "main" / "nearest_neighbours.json").read_text())
    nn3a = json.loads((C.TASK3_OUT / "main_abtt" / "nearest_neighbours.json").read_text())
    pd.DataFrame([dict(word=w, T1=", ".join(x for x, _ in nn1.get(w, [])[:6]),
                       T3=", ".join(x for x, _ in nn3.get(w, [])[:6]),
                       T3_abtt=", ".join(x for x, _ in nn3a.get(w, [])[:6])) for w in nn3]).to_csv(
        QUAL / "nearest_neighbours.csv", index=False)
    a1 = pd.read_csv(t1_dir() / "analogy_examples.csv")
    a3 = pd.read_csv(C.TASK3_OUT / "main" / "analogy_examples.csv")
    m = a3.merge(a1, on=["section", "a", "b", "c", "gold"], suffixes=("_T3", "_T1"))
    pick = pd.concat([m[m.correct_T3 & ~m.correct_T1].groupby("section").head(2),
                      m[~m.correct_T3 & m.correct_T1].head(6),
                      m[~m.correct_T3 & ~m.correct_T1].groupby("section").head(1).head(12)])
    pick.to_csv(QUAL / "analogy_examples.csv", index=False)


def main() -> None:
    QUAL.mkdir(parents=True, exist_ok=True)
    L = best_layer()["l_star"]
    sw = pd.read_csv(EXT / "layer_sweep.csv")
    fig_layer_sweep(sw, L)
    fig_ladder(pd.read_csv(EXT / "ladder.csv"))
    fig_ladder_relations(pd.read_csv(EXT / "ladder_relations.csv"))
    fig_ws_by_frequency({"T1 PPMI-SVD": json.loads((t1_dir() / "results.json").read_text()),
                         f"T3 BERT-static L{L}": json.loads((C.TASK3_OUT / "main" / "results.json").read_text()),
                         f"T3 + ABTT": json.loads((C.TASK3_OUT / "main_abtt" / "results.json").read_text())})
    ex = fig_mst_example(json.loads((EXT / "ladder_20_edges.json").read_text()))
    tables(L)
    (QUAL / "mst_example.json").write_text(json.dumps(ex, indent=1))
    print("[plot3] figures ->", P2.OUT, "| tables ->", QUAL)
    print("[plot3] MST example:", ex)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code/task3/pool_static.py
"""Task 3a: BERT-derived *static* word vectors.

For every word type w of the Task 1 vocabulary (lowercased train types, same order, so row i of
both matrices is the same word):

    v_w(layer) = mean over all train occurrences of w of the layer-`layer` BERT word vector

The word vectors are the cached Task 2 features (each = mean of its wordpieces, computed IN
CONTEXT, i.e. the whole sentence was fed to BERT). Averaging over all contexts removes the
context and leaves one vector per type, which is what Bommasani et al. (2020) call a pooled
static embedding (mean pooling).

The required run is layer l* (from outputs/task2/best_layer.json). By default all 13 layers are
pooled, because the Tier B layer sweep (extensions_task3.py) needs them; it is cheap (one sparse
matrix product per layer).

Usage (from the project root):
    python code/task3/pool_static.py               # all layers 0..12
    python code/task3/pool_static.py --layers 8    # only l*

Outputs -> outputs/task3/static/
    layer{L}.npy   float32 [V, 768]
    counts.npy     int64 [V]   number of train occurrences (contexts averaged) per type
    meta.json      vocab source, layers, sanity checks
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import scipy.sparse as sp

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common import config as C  # noqa: E402
from common.data import load_ewt  # noqa: E402
from common.features import load_layer, load_meta, load_offsets  # noqa: E402

STATIC_DIR = C.TASK3_OUT / "static"


def load_t1_vocab() -> tuple[list[str], dict[str, int]]:
    vocab = json.loads((C.TASK1_OUT / "vocab.json").read_text())
    return vocab, {w: i for i, w in enumerate(vocab)}


def pooling_matrix(train, w2i) -> tuple[sp.csr_matrix, np.ndarray]:
    """Sparse [V, N_words] matrix with 1/count(w) at (w, every occurrence of w)."""
    ids = np.array([w2i[w] for s in train for w in s.lower], dtype=np.int64)
    counts = np.bincount(ids, minlength=len(w2i))
    assert (counts > 0).all(), "every T1 vocab type occurs in train"
    S = sp.csr_matrix((1.0 / counts[ids], (ids, np.arange(len(ids)))), shape=(len(w2i), len(ids)),
                      dtype=np.float64)
    return S, counts


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--layers", type=int, nargs="+", default=list(range(C.BERT_NUM_LAYERS + 1)))
    args = ap.parse_args()
    STATIC_DIR.mkdir(parents=True, exist_ok=True)

    vocab, w2i = load_t1_vocab()
    train = load_ewt("train")
    meta = load_meta("train")
    offsets = load_offsets("train")
    assert meta["sent_ids"] == [s.sent_id for s in train], "feature order != EWT order"
    assert int(offsets[-1]) == sum(len(s) for s in train), "feature rows != train tokens"
    S, counts = pooling_matrix(train, w2i)
    np.save(STATIC_DIR / "counts.npy", counts)
    print(f"[pool] vocab={len(vocab)} train words={S.shape[1]} singletons={(counts == 1).sum()}", flush=True)

    w = w2i["good"]  # sanity check word: its pooled vector must equal the plain mean of its rows
    rows = np.flatnonzero(np.array([x == "good" for s in train for x in s.lower]))
    checks = {}
    for layer in args.layers:
        t0 = time.time()
        X = np.asarray(load_layer("train", layer, mmap=False), dtype=np.float32)
        V = np.asarray(S @ X.astype(np.float64), dtype=np.float32)  # float64 accumulation
        err = float(np.abs(V[w] - X[rows].astype(np.float64).mean(0)).max())
        assert np.isfinite(V).all() and err < 1e-4, f"pooling check failed (err={err})"
        np.save(STATIC_DIR / f"layer{layer}.npy", V)
        checks[layer] = dict(max_abs_err_check=err, mean_norm=float(np.linalg.norm(V, axis=1).mean()))
        print(f"[pool] layer {layer:2d}: [{V.shape[0]}, {V.shape[1]}] mean |v|={checks[layer]['mean_norm']:.2f} "
              f"check err={err:.1e} ({time.time() - t0:.1f} s)", flush=True)
        del X, V

    (STATIC_DIR / "meta.json").write_text(json.dumps(dict(
        vocab="outputs/task1/vocab.json (same row order as the T1 vectors)",
        pooling="mean over all train occurrences of the in-context word vector (word = mean of wordpieces)",
        model=C.BERT_MODEL, revision=C.BERT_REVISION, layers=args.layers, checks=checks), indent=1))
    print("[pool] done ->", STATIC_DIR)


if __name__ == "__main__":
    main()


In [ ]:
run("code/run_all.py", "--tasks", "3")

## 6. Results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
bl = json.load(open("outputs/task2/best_layer.json"))
display(Markdown(f"**Task 2:** best layer ℓ* = {bl['l_star']} (chosen on dev); test UUAS = {100*bl['test']['uuas']:.1f}%, "
                 f"depth Spearman = {bl['test']['nspr']:.3f}"))
display(pd.read_csv("outputs/task2/paper_comparison.csv"))
display(Markdown("**Task 1 vs Task 3** (identical items; the Task 2 probe as a reference)"))
display(pd.read_csv("outputs/task3/comparison_table.csv")[["system", "ws_rho_cov", "ws_rho_full", "an_acc_cov", "an_acc_full", "mst20_uuas", "pool_uuas"]])
display(Markdown("**Diagnostic ladder** (the same 20 sentences / 660-sentence pool)"))
display(pd.read_csv("outputs/task3/extensions/ladder.csv")[["system", "mst20_uuas", "pool_uuas"]])

In [ ]:
for f in ["outputs/task2/figures/metrics_vs_layer.png", "outputs/task3/figures/layer_sweep.png",
          "outputs/task3/figures/ladder.png", "outputs/task2/figures/example_typical.png",
          "outputs/task3/figures/mst_example.png"]:
    display(Markdown(f"`{f}`")); display(Image(f, width=850))

All result files are in `outputs/` (see the table of output folders in the project README).